# Experiment 3 — Targeted Fixed-Width Hotspot Repair

## Experiment 3A — Intervention Pilot

### Research Question

Given a fixed-width GidNET reuse plan \(P\), can a legal local modification
\(P'\) that reduces quotient-interaction-graph load hotspots also reduce
physical routing cost?

We require:

\[
width(P') = width(P)
\]

The candidate repair is selected **without observing routed SWAP count or
routed depth**.

Primary structural objectives:

1. minimize `max_weighted_degree`
2. break ties using `weighted_degree_cv`

The first repair operator studied is a legal two-chain suffix swap.

In [1]:
from pathlib import Path
import ast
import json
import inspect
import math
import random

import numpy as np
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

ROUTING_SEEDS = [42, 43, 44, 45, 46]
LOGICAL_DEPTH_TOLERANCE = 0.05

print("Experiment 3A setup ready.")
print("Routing seeds:", ROUTING_SEEDS)
print("Logical-depth tolerance:", LOGICAL_DEPTH_TOLERANCE)

Experiment 3A setup ready.
Routing seeds: [42, 43, 44, 45, 46]
Logical-depth tolerance: 0.05


In [2]:
# ------------------------------------------------------------
# Locate Experiment 1 / Experiment 2 result files
# ------------------------------------------------------------

ROOT = Path.cwd()

def find_file(filename):
    matches = list(ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} anywhere under:\n{ROOT}"
        )

    if len(matches) > 1:
        print(f"[WARNING] Multiple copies of {filename} found:")
        for p in matches:
            print("   ", p)

    return matches[0]


EXP1_RAW_PATH = find_file("experiment1_raw.csv")
EXP1_PLAN_PATH = find_file("experiment1_plan_summary.csv")
EXP2_PLAN_PATH = find_file("experiment2_plan_features.csv")

print("Experiment 1 raw:")
print(" ", EXP1_RAW_PATH)

print("\nExperiment 1 plan summary:")
print(" ", EXP1_PLAN_PATH)

print("\nExperiment 2 plan features:")
print(" ", EXP2_PLAN_PATH)

Experiment 1 raw:
  C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment1_results\experiment1_raw.csv

Experiment 1 plan summary:
  C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment1_results\experiment1_plan_summary.csv

Experiment 2 plan features:
  C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment2_results\experiment2_plan_features.csv


In [3]:
# ------------------------------------------------------------
# Load previous experiment outputs
# ------------------------------------------------------------

exp1_raw = pd.read_csv(EXP1_RAW_PATH)
exp1_plans = pd.read_csv(EXP1_PLAN_PATH)
exp2_plans = pd.read_csv(EXP2_PLAN_PATH)

print("Shapes")
print("------")
print("Experiment 1 raw:         ", exp1_raw.shape)
print("Experiment 1 plan summary:", exp1_plans.shape)
print("Experiment 2 plan features:", exp2_plans.shape)

print("\nExperiment 1 raw columns:")
print(exp1_raw.columns.tolist())

print("\nExperiment 1 plan-summary columns:")
print(exp1_plans.columns.tolist())

print("\nExperiment 2 plan-feature columns:")
print(exp2_plans.columns.tolist())

Shapes
------
Experiment 1 raw:          (250, 13)
Experiment 1 plan summary: (50, 17)
Experiment 2 plan features: (50, 41)

Experiment 1 raw columns:
['circuit', 'plan_id', 'gidnet_seed', 'plan_signature', 'width', 'raw_logical_depth', 'normalized_logical_depth', 'measurements', 'conditional_x', 'routing_seed', 'swap_count', 'routed_depth', 'cx_count']

Experiment 1 plan-summary columns:
['circuit', 'plan_id', 'gidnet_seed', 'plan_signature', 'width', 'raw_logical_depth', 'normalized_logical_depth', 'mean_swaps', 'std_swaps', 'min_swaps', 'max_swaps', 'mean_routed_depth', 'std_routed_depth', 'min_routed_depth', 'max_routed_depth', 'mean_cx', 'num_routing_trials']

Experiment 2 plan-feature columns:
['circuit', 'plan_id', 'gidnet_seed', 'width', 'num_reuse_boundaries', 'normalized_logical_depth', 'mean_swaps', 'mean_routed_depth', 'quotient_unique_edges', 'quotient_density', 'max_degree', 'max_weighted_degree', 'max_edge_weight', 'mean_raw_jaccard_incompat', 'max_raw_jaccard_incompat',

In [4]:
# ------------------------------------------------------------
# Verify Experiment-2 hotspot metrics
# ------------------------------------------------------------

required_exp2_columns = [
    "circuit",
    "plan_id",
    "max_weighted_degree",
    "weighted_degree_cv",
]

missing = [
    col for col in required_exp2_columns
    if col not in exp2_plans.columns
]

if missing:
    print("Missing required Experiment-2 columns:")
    print(missing)
else:
    print("All primary Experiment-2 hotspot columns are present.")

    display(
        exp2_plans[
            required_exp2_columns
        ].head(10)
    )

All primary Experiment-2 hotspot columns are present.


,circuit,plan_id,max_weighted_degree,weighted_degree_cv
0,system9.qasm,P0,63.0,0.438558
1,system9.qasm,P1,71.0,0.485364
2,system9.qasm,P10,63.0,0.432846
3,system9.qasm,P11,65.0,0.383898
4,system9.qasm,P12,71.0,0.506689
5,system9.qasm,P2,77.0,0.517623
6,system9.qasm,P3,67.0,0.475324
7,system9.qasm,P4,63.0,0.384505
8,system9.qasm,P5,63.0,0.383290
9,system9.qasm,P6,63.0,0.433923


In [5]:
# ------------------------------------------------------------
# Inspect one real reuse plan
# ------------------------------------------------------------

example_row = exp2_plans[
    (exp2_plans["circuit"] == "system9.qasm") &
    (exp2_plans["plan_id"] == "P0")
].iloc[0]

print("Circuit:", example_row["circuit"])
print("Plan ID:", example_row["plan_id"])
print("Width:", example_row["width"])
print("Max weighted degree:", example_row["max_weighted_degree"])
print("Weighted-degree CV:", example_row["weighted_degree_cv"])

print("\nRaw reuse_chains:")
print(example_row["reuse_chains"])

print("\nRaw reuse_boundaries:")
print(example_row["reuse_boundaries"])

print("\nPlan signature:")
print(example_row["plan_signature"])

Circuit: system9.qasm
Plan ID: P0
Width: 7
Max weighted degree: 63.0
Weighted-degree CV: 0.4385582439463809

Raw reuse_chains:
((0, 11), (1,), (2, 4, 6), (3, 7, 8), (5,), (9,), (10,))

Raw reuse_boundaries:
((0, 11), (2, 4), (4, 6), (3, 7), (7, 8))

Plan signature:
((0, 11), (1,), (2, 4, 6), (3, 7, 8), (5,), (9,), (10,))


In [6]:
# ------------------------------------------------------------
# Parse reuse-plan representation saved by Experiment 2
# ------------------------------------------------------------

def parse_saved_structure(value):
    if isinstance(value, (list, tuple, dict)):
        return value

    if pd.isna(value):
        return None

    return ast.literal_eval(str(value))


example_chains = parse_saved_structure(example_row["reuse_chains"])
example_boundaries = parse_saved_structure(example_row["reuse_boundaries"])

print("Parsed reuse chains:")
for i, chain in enumerate(example_chains):
    print(f"  Chain {i}: {chain}")

print("\nParsed reuse boundaries:")
for boundary in example_boundaries:
    print(" ", boundary)

print("\nNumber of chains:", len(example_chains))
print("Saved width:", int(example_row["width"]))

assert len(example_chains) == int(example_row["width"])

print("\n✓ Number of chains equals reuse width.")

Parsed reuse chains:
  Chain 0: (0, 11)
  Chain 1: (1,)
  Chain 2: (2, 4, 6)
  Chain 3: (3, 7, 8)
  Chain 4: (5,)
  Chain 5: (9,)
  Chain 6: (10,)

Parsed reuse boundaries:
  (0, 11)
  (2, 4)
  (4, 6)
  (3, 7)
  (7, 8)

Number of chains: 7
Saved width: 7

✓ Number of chains equals reuse width.


In [7]:
# ------------------------------------------------------------
# Verify that saved reuse_boundaries agree with reuse_chains
# ------------------------------------------------------------

def boundaries_from_chains(chains):
    return [
        (chain[i], chain[i + 1])
        for chain in chains
        for i in range(len(chain) - 1)
    ]


derived_boundaries = boundaries_from_chains(example_chains)

print("Derived boundaries:")
print(derived_boundaries)

print("\nSaved boundaries:")
print(example_boundaries)

saved_boundary_set = {tuple(x) for x in example_boundaries}
derived_boundary_set = {tuple(x) for x in derived_boundaries}

assert saved_boundary_set == derived_boundary_set

print("\n✓ reuse_chains and reuse_boundaries are consistent.")

Derived boundaries:
[(0, 11), (2, 4), (4, 6), (3, 7), (7, 8)]

Saved boundaries:
((0, 11), (2, 4), (4, 6), (3, 7), (7, 8))

✓ reuse_chains and reuse_boundaries are consistent.


In [8]:
# ------------------------------------------------------------
# Load the original benchmark circuit
# ------------------------------------------------------------

from pathlib import Path
from qiskit import qasm2
from gidnet.qubitreuse import GidNET

BENCH_DIR = Path(
    r"C:\Users\ASUS\Documents\Quantum computing\Qiskit"
    r"\CaQR-main\CaQR-main\benchmarks"
)

print("Benchmark directory:", BENCH_DIR)
print("Exists:", BENCH_DIR.exists())

example_circuit_path = BENCH_DIR / example_row["circuit"]

if not example_circuit_path.exists():
    raise FileNotFoundError(
        f"Could not find:\n{example_circuit_path}"
    )

print("\nCircuit path:")
print(example_circuit_path)

original_circuit = qasm2.load(str(example_circuit_path))

gid = GidNET(original_circuit)

C0 = np.array(gid.candidate_matrix_copy, copy=True)

print("\nOriginal logical qubits:", original_circuit.num_qubits)
print("Candidate matrix shape:", C0.shape)
print(
    "Number of initially legal reuse relations:",
    int(C0.sum())
)

Benchmark directory: C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks
Exists: True

Circuit path:
C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks\system9.qasm

Original logical qubits: 16
Candidate matrix shape: (32, 32)
Number of initially legal reuse relations: 885


In [9]:
# ------------------------------------------------------------
# Check existing reuse boundaries against GidNET candidate matrix
# ------------------------------------------------------------

boundary_checks = []

for a, b in derived_boundaries:
    candidate_value = int(C0[a, b])

    boundary_checks.append({
        "from_qubit": a,
        "to_qubit": b,
        "candidate_matrix_value": candidate_value,
        "initially_legal": candidate_value == 1,
    })

boundary_checks_df = pd.DataFrame(boundary_checks)

display(boundary_checks_df)

print(
    "\nAll existing boundaries initially legal:",
    bool(boundary_checks_df["initially_legal"].all())
)

,from_qubit,to_qubit,candidate_matrix_value,initially_legal
0,0,11,1,True
1,2,4,1,True
2,4,6,1,True
3,3,7,1,True
4,7,8,1,True



All existing boundaries initially legal: True


In [10]:
active_qubits = set()

for item in original_circuit.data:
    if item.operation.name in {"barrier", "measure"}:
        continue

    for q in item.qubits:
        active_qubits.add(
            original_circuit.find_bit(q).index
        )

active_qubits = sorted(active_qubits)

saved_plan_qubits = sorted(
    q
    for chain in example_chains
    for q in chain
)

print("Declared qubits:", original_circuit.num_qubits)
print("Active logical qubits:", active_qubits)
print("Qubits in saved P0:", saved_plan_qubits)

print(
    "Same active-qubit set?",
    active_qubits == saved_plan_qubits
)

print(
    "Inactive declared qubits:",
    sorted(
        set(range(original_circuit.num_qubits))
        - set(active_qubits)
    )
)

Declared qubits: 16
Active logical qubits: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Qubits in saved P0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Same active-qubit set? True
Inactive declared qubits: [12, 13, 14, 15]


In [11]:
# ------------------------------------------------------------
# Expand an active reuse plan with inactive singleton qubits
# ------------------------------------------------------------

def expand_with_inactive_singletons(circuit, active_chains):
    used = {
        q
        for chain in active_chains
        for q in chain
    }

    inactive = [
        q for q in range(circuit.num_qubits)
        if q not in used
    ]

    full_chains = [
        list(chain)
        for chain in active_chains
    ]

    full_chains.extend([[q] for q in inactive])

    return full_chains


example_full_chains = expand_with_inactive_singletons(
    original_circuit,
    example_chains
)

print("Active plan:")
for chain in example_chains:
    print(" ", tuple(chain))

print("\nFull GidNET plan:")
for chain in example_full_chains:
    print(" ", tuple(chain))

print("\nActive width:", len(example_chains))
print("Full declared-chain count:", len(example_full_chains))

Active plan:
  (0, 11)
  (1,)
  (2, 4, 6)
  (3, 7, 8)
  (5,)
  (9,)
  (10,)

Full GidNET plan:
  (0, 11)
  (1,)
  (2, 4, 6)
  (3, 7, 8)
  (5,)
  (9,)
  (10,)
  (12,)
  (13,)
  (14,)
  (15,)

Active width: 7
Full declared-chain count: 11


In [12]:
# ------------------------------------------------------------
# Full structural validator for an ACTIVE reuse plan
# ------------------------------------------------------------

import copy
from qiskit.dagcircuit.dagnode import DAGOpNode


def validate_active_plan(circuit, active_chains):
    """
    Validate an active reuse plan against the original circuit.

    active_chains:
        only chains containing active logical qubits,
        exactly as stored by Experiments 1/2.
    """

    # --------------------------------------------------------
    # 1. Find active logical qubits
    # --------------------------------------------------------

    active = set()

    for item in circuit.data:
        if item.operation.name in {"barrier", "measure"}:
            continue

        for q in item.qubits:
            active.add(
                circuit.find_bit(q).index
            )

    # --------------------------------------------------------
    # 2. Every active qubit must appear exactly once
    # --------------------------------------------------------

    flat = [
        q
        for chain in active_chains
        for q in chain
    ]

    if len(flat) != len(set(flat)):
        return {
            "legal": False,
            "reason": "duplicate_logical_qubit",
        }

    if set(flat) != active:
        return {
            "legal": False,
            "reason": "active_qubit_set_mismatch",
            "missing": sorted(active - set(flat)),
            "extra": sorted(set(flat) - active),
        }

    # --------------------------------------------------------
    # 3. Add inactive declared qubits as singleton chains
    # --------------------------------------------------------

    full_chains = expand_with_inactive_singletons(
        circuit,
        active_chains
    )

    # --------------------------------------------------------
    # 4. Fresh GidNET candidate matrix
    # --------------------------------------------------------

    test_gid = GidNET(
        copy.deepcopy(circuit)
    )

    C = np.array(
        test_gid.candidate_matrix_copy,
        copy=True
    )

    boundaries = boundaries_from_chains(
        active_chains
    )

    illegal_boundaries = []

    for a, b in boundaries:
        if C[a, b] != 1:
            illegal_boundaries.append((a, b))

    if illegal_boundaries:
        return {
            "legal": False,
            "reason": "boundary_not_in_candidate_matrix",
            "illegal_boundaries": illegal_boundaries,
        }

    # --------------------------------------------------------
    # 5. Apply COMPLETE plan using GidNET's own DAG logic
    # --------------------------------------------------------

    test_gid.qubit_reuse_sequences = copy.deepcopy(
        full_chains
    )

    try:
        test_gid.add_qubit_reuse_edges()

    except Exception as e:
        return {
            "legal": False,
            "reason": "add_reuse_edges_failed",
            "error": repr(e),
        }

    # Same barrier handling as compile_to_dynamic_circuit()
    barrier_nodes = [
        node
        for node in list(
            test_gid.circuit_dag_with_reuse_edges.nodes()
        )
        if isinstance(node, DAGOpNode)
        and node.op.name == "barrier"
    ]

    for node in barrier_nodes:
        test_gid.circuit_dag_with_reuse_edges.remove_op_node(
            node
        )

    # --------------------------------------------------------
    # 6. Full DAG-cycle check
    # --------------------------------------------------------

    try:
        all_nodes = list(
            test_gid.circuit_dag_with_reuse_edges.nodes()
        )

        topo_nodes = list(
            test_gid.circuit_dag_with_reuse_edges.topological_nodes()
        )

    except Exception as e:
        return {
            "legal": False,
            "reason": "topological_sort_failed",
            "error": repr(e),
        }

    if len(all_nodes) != len(topo_nodes):
        return {
            "legal": False,
            "reason": "dependency_cycle",
        }

    return {
        "legal": True,
        "reason": "ok",
        "active_width": len(active_chains),
        "full_chain_count": len(full_chains),
        "num_reuse_boundaries": len(boundaries),
    }

In [13]:
validation = validate_active_plan(
    original_circuit,
    example_chains
)

print(validation)

{'legal': True, 'reason': 'ok', 'active_width': 7, 'full_chain_count': 11, 'num_reuse_boundaries': 5}


In [14]:
# ------------------------------------------------------------
# Reproduce Experiment-2 quotient interaction-load metrics
# ------------------------------------------------------------

from collections import defaultdict


def compute_quotient_load_metrics(circuit, chains):
    """
    Build the quotient interaction graph induced by reuse chains.

    Each reuse chain is one quotient node.
    Every original two-qubit interaction between logical qubits
    belonging to different chains increments the corresponding
    quotient-edge weight by one.
    """

    # logical qubit -> reuse-chain index
    qubit_to_chain = {}

    for chain_idx, chain in enumerate(chains):
        for q in chain:
            if q in qubit_to_chain:
                raise ValueError(
                    f"Logical qubit {q} appears in multiple chains."
                )

            qubit_to_chain[q] = chain_idx

    # Weighted undirected quotient edges
    edge_weights = defaultdict(int)

    for item in circuit.data:

        qargs = list(item.qubits)

        # Only two-qubit interactions
        if len(qargs) != 2:
            continue

        q0 = circuit.find_bit(qargs[0]).index
        q1 = circuit.find_bit(qargs[1]).index

        # Ignore declared-but-inactive qubits not represented
        # in the active reuse plan.
        if (
            q0 not in qubit_to_chain
            or q1 not in qubit_to_chain
        ):
            continue

        c0 = qubit_to_chain[q0]
        c1 = qubit_to_chain[q1]

        # Interaction inside the same reused wire does not
        # create an inter-chain quotient edge.
        if c0 == c1:
            continue

        edge = tuple(sorted((c0, c1)))
        edge_weights[edge] += 1

    # --------------------------------------------------------
    # Weighted degree of every quotient node
    # --------------------------------------------------------

    weighted_degree = np.zeros(
        len(chains),
        dtype=float
    )

    for (u, v), weight in edge_weights.items():
        weighted_degree[u] += weight
        weighted_degree[v] += weight

    max_weighted_degree = (
        float(weighted_degree.max())
        if len(weighted_degree)
        else 0.0
    )

    mean_wd = float(weighted_degree.mean())

    weighted_degree_cv = (
        float(weighted_degree.std(ddof=0) / mean_wd)
        if mean_wd > 0
        else 0.0
    )

    hotspot_indices = [
        i
        for i, value in enumerate(weighted_degree)
        if np.isclose(
            value,
            max_weighted_degree
        )
    ]

    return {
        "edge_weights": dict(edge_weights),
        "weighted_degree": weighted_degree,
        "max_weighted_degree": max_weighted_degree,
        "weighted_degree_cv": weighted_degree_cv,
        "hotspot_indices": hotspot_indices,
    }

In [15]:
p0_metrics = compute_quotient_load_metrics(
    original_circuit,
    example_chains
)

print("Weighted degree per reuse chain:")

for idx, value in enumerate(
    p0_metrics["weighted_degree"]
):
    print(
        f"Chain {idx}: "
        f"{example_chains[idx]} "
        f"-> weighted degree = {value}"
    )

print("\nRecomputed max_weighted_degree:")
print(p0_metrics["max_weighted_degree"])

print("Saved Experiment-2 max_weighted_degree:")
print(example_row["max_weighted_degree"])

print("\nRecomputed weighted_degree_cv:")
print(p0_metrics["weighted_degree_cv"])

print("Saved Experiment-2 weighted_degree_cv:")
print(example_row["weighted_degree_cv"])

print(
    "\nHotspot chain index/indices:",
    p0_metrics["hotspot_indices"]
)

Weighted degree per reuse chain:
Chain 0: (0, 11) -> weighted degree = 31.0
Chain 1: (1,) -> weighted degree = 13.0
Chain 2: (2, 4, 6) -> weighted degree = 61.0
Chain 3: (3, 7, 8) -> weighted degree = 50.0
Chain 4: (5,) -> weighted degree = 23.0
Chain 5: (9,) -> weighted degree = 63.0
Chain 6: (10,) -> weighted degree = 59.0

Recomputed max_weighted_degree:
63.0
Saved Experiment-2 max_weighted_degree:
63.0

Recomputed weighted_degree_cv:
0.43855824394638093
Saved Experiment-2 weighted_degree_cv:
0.4385582439463809

Hotspot chain index/indices: [5]


In [16]:
assert np.isclose(
    p0_metrics["max_weighted_degree"],
    float(example_row["max_weighted_degree"])
)

assert np.isclose(
    p0_metrics["weighted_degree_cv"],
    float(example_row["weighted_degree_cv"])
)

print(
    "\n✓ Quotient metrics exactly reproduce "
    "Experiment 2 for system9 / P0."
)


✓ Quotient metrics exactly reproduce Experiment 2 for system9 / P0.


In [17]:
# ------------------------------------------------------------
# Inspect the current interaction-load hotspot
# ------------------------------------------------------------

hotspot_indices = p0_metrics["hotspot_indices"]

print("Hotspot chain(s):")

for idx in hotspot_indices:
    print(
        f"  Chain {idx}: {example_chains[idx]}"
        f" | weighted_degree="
        f"{p0_metrics['weighted_degree'][idx]}"
    )

print(
    "\nSeed max_weighted_degree:",
    p0_metrics["max_weighted_degree"]
)

print(
    "Seed weighted_degree_cv:",
    p0_metrics["weighted_degree_cv"]
)

Hotspot chain(s):
  Chain 5: (9,) | weighted_degree=63.0

Seed max_weighted_degree: 63.0
Seed weighted_degree_cv: 0.43855824394638093


In [18]:
# ------------------------------------------------------------
# Two-chain suffix-swap neighborhood
# ------------------------------------------------------------

def canonical_plan_signature(chains):
    """
    Used only for deduplication.
    Internal order of each chain is preserved.
    Chain ordering itself is canonicalized.
    """

    return tuple(
        sorted(
            (tuple(chain) for chain in chains)
        )
    )


def generate_hotspot_suffix_swap_neighbors(
    circuit,
    seed_chains,
    hotspot_indices,
):
    """
    Generate legal two-chain suffix-swap neighbors involving
    at least one current hotspot chain.

    Both chains must contain at least two logical lifetimes,
    because the cut must split each chain into a non-empty
    prefix and a non-empty suffix.
    """

    seed_chains = [
        tuple(chain)
        for chain in seed_chains
    ]

    seed_signature = canonical_plan_signature(
        seed_chains
    )

    seen = {seed_signature}
    legal_neighbors = []

    rejection_reasons = defaultdict(int)

    generated_raw = 0

    n = len(seed_chains)

    for hotspot_idx in hotspot_indices:

        A = seed_chains[hotspot_idx]

        # Need non-empty prefix and suffix
        if len(A) < 2:
            continue

        for other_idx in range(n):

            if other_idx == hotspot_idx:
                continue

            B = seed_chains[other_idx]

            if len(B) < 2:
                continue

            # Cut is AFTER positions:
            #
            # A[:cut_A] | A[cut_A:]
            #
            # Therefore cut positions are:
            # 1 .. len(A)-1

            for cut_A in range(1, len(A)):

                for cut_B in range(1, len(B)):

                    generated_raw += 1

                    prefix_A = A[:cut_A]
                    suffix_A = A[cut_A:]

                    prefix_B = B[:cut_B]
                    suffix_B = B[cut_B:]

                    new_A = (
                        prefix_A + suffix_B
                    )

                    new_B = (
                        prefix_B + suffix_A
                    )

                    candidate = [
                        tuple(chain)
                        for chain in seed_chains
                    ]

                    candidate[hotspot_idx] = new_A
                    candidate[other_idx] = new_B

                    # ----------------------------------------
                    # Preserve active width
                    # ----------------------------------------

                    if len(candidate) != len(seed_chains):
                        rejection_reasons[
                            "width_changed"
                        ] += 1
                        continue

                    # ----------------------------------------
                    # Deduplicate
                    # ----------------------------------------

                    signature = (
                        canonical_plan_signature(candidate)
                    )

                    if signature in seen:
                        rejection_reasons[
                            "duplicate"
                        ] += 1
                        continue

                    seen.add(signature)

                    # ----------------------------------------
                    # Full legality validation
                    # ----------------------------------------

                    validation = validate_active_plan(
                        circuit,
                        candidate
                    )

                    if not validation["legal"]:
                        rejection_reasons[
                            validation["reason"]
                        ] += 1
                        continue

                    # ----------------------------------------
                    # Structural diff
                    # ----------------------------------------

                    before_boundaries = set(
                        boundaries_from_chains(
                            seed_chains
                        )
                    )

                    after_boundaries = set(
                        boundaries_from_chains(
                            candidate
                        )
                    )

                    added_boundaries = sorted(
                        after_boundaries
                        - before_boundaries
                    )

                    removed_boundaries = sorted(
                        before_boundaries
                        - after_boundaries
                    )

                    legal_neighbors.append({
                        "hotspot_chain_idx":
                            hotspot_idx,

                        "other_chain_idx":
                            other_idx,

                        "cut_hotspot":
                            cut_A,

                        "cut_other":
                            cut_B,

                        "candidate_chains":
                            candidate,

                        "plan_signature":
                            signature,

                        "added_boundaries":
                            added_boundaries,

                        "removed_boundaries":
                            removed_boundaries,

                        "validation":
                            validation,
                    })

    return {
        "generated_raw": generated_raw,
        "legal_neighbors": legal_neighbors,
        "rejection_reasons": dict(
            rejection_reasons
        ),
    }

In [19]:
p0_neighborhood = (
    generate_hotspot_suffix_swap_neighbors(
        original_circuit,
        example_chains,
        hotspot_indices,
    )
)

print(
    "Raw suffix-swap attempts:",
    p0_neighborhood["generated_raw"]
)

print(
    "Unique legal neighbors:",
    len(
        p0_neighborhood[
            "legal_neighbors"
        ]
    )
)

print("\nRejected candidates:")
print(
    p0_neighborhood[
        "rejection_reasons"
    ]
)

Raw suffix-swap attempts: 0
Unique legal neighbors: 0

Rejected candidates:
{}


In [20]:
for i, neighbor in enumerate(
    p0_neighborhood["legal_neighbors"]
):

    print(
        "\n"
        + "=" * 70
    )

    print(f"Neighbor N{i}")

    print(
        "Hotspot chain:",
        neighbor["hotspot_chain_idx"]
    )

    print(
        "Other chain:",
        neighbor["other_chain_idx"]
    )

    print(
        "Cuts:",
        neighbor["cut_hotspot"],
        neighbor["cut_other"]
    )

    print("\nCandidate chains:")

    for chain in neighbor[
        "candidate_chains"
    ]:
        print(" ", chain)

    print(
        "\nRemoved boundaries:",
        neighbor["removed_boundaries"]
    )

    print(
        "Added boundaries:",
        neighbor["added_boundaries"]
    )

In [22]:
# ------------------------------------------------------------
# Diagnose strict suffix-swap applicability across ALL 50 plans
# ------------------------------------------------------------

circuit_cache = {}
applicability_rows = []

for _, row in exp2_plans.iterrows():

    circuit_name = row["circuit"]

    # --------------------------------------------
    # Load circuit once
    # --------------------------------------------

    if circuit_name not in circuit_cache:

        circuit_path = BENCH_DIR / circuit_name

        if not circuit_path.exists():
            raise FileNotFoundError(
                f"Missing benchmark: {circuit_path}"
            )

        circuit_cache[circuit_name] = qasm2.load(
            str(circuit_path)
        )

    qc = circuit_cache[circuit_name]

    # --------------------------------------------
    # Parse saved reuse chains
    # --------------------------------------------

    chains = parse_saved_structure(
        row["reuse_chains"]
    )

    chains = [
        tuple(chain)
        for chain in chains
    ]

    # --------------------------------------------
    # Recompute Experiment-2 quotient metrics
    # --------------------------------------------

    metrics = compute_quotient_load_metrics(
        qc,
        chains
    )

    # Strong consistency check
    assert np.isclose(
        metrics["max_weighted_degree"],
        float(row["max_weighted_degree"])
    )

    assert np.isclose(
        metrics["weighted_degree_cv"],
        float(row["weighted_degree_cv"])
    )

    hotspot_indices = metrics[
        "hotspot_indices"
    ]

    hotspot_lengths = [
        len(chains[i])
        for i in hotspot_indices
    ]

    # --------------------------------------------
    # Count RAW strict suffix-swap attempts
    #
    # strict definition:
    # both chains must have a non-empty prefix
    # AND a non-empty suffix
    # --------------------------------------------

    raw_attempts = 0

    for h in hotspot_indices:

        A = chains[h]

        if len(A) < 2:
            continue

        for j, B in enumerate(chains):

            if j == h:
                continue

            if len(B) < 2:
                continue

            raw_attempts += (
                (len(A) - 1)
                * (len(B) - 1)
            )

    applicability_rows.append({
        "circuit":
            circuit_name,

        "plan_id":
            row["plan_id"],

        "width":
            int(row["width"]),

        "hotspot_indices":
            tuple(hotspot_indices),

        "hotspot_chains":
            tuple(
                chains[i]
                for i in hotspot_indices
            ),

        "hotspot_lengths":
            tuple(hotspot_lengths),

        "max_weighted_degree":
            metrics[
                "max_weighted_degree"
            ],

        "weighted_degree_cv":
            metrics[
                "weighted_degree_cv"
            ],

        "raw_strict_suffix_swap_attempts":
            raw_attempts,

        "structurally_applicable":
            raw_attempts > 0,
    })


applicability_df = pd.DataFrame(
    applicability_rows
)

print(
    "Total seed plans:",
    len(applicability_df)
)

print(
    "Structurally applicable:",
    int(
        applicability_df[
            "structurally_applicable"
        ].sum()
    )
)

print(
    "No strict suffix-swap possible:",
    int(
        (~applicability_df[
            "structurally_applicable"
        ]).sum()
    )
)

print("\nPer circuit:")

display(
    applicability_df
    .groupby("circuit")[
        "structurally_applicable"
    ]
    .agg(
        total="size",
        applicable="sum"
    )
    .assign(
        inapplicable=lambda x:
            x["total"] - x["applicable"]
    )
)

print(
    "\nPlans where strict hotspot suffix-swap "
    "is structurally possible:"
)

display(
    applicability_df[
        applicability_df[
            "structurally_applicable"
        ]
    ][
        [
            "circuit",
            "plan_id",
            "hotspot_chains",
            "hotspot_lengths",
            "max_weighted_degree",
            "raw_strict_suffix_swap_attempts",
        ]
    ]
)

Total seed plans: 50
Structurally applicable: 41
No strict suffix-swap possible: 9

Per circuit:


,total,applicable,inapplicable
circuit,,,
alu-bdd_288.qasm,6,2,4
mini_alu_305.qasm,16,16,0
rd73_140.qasm,6,6,0
sym9_146.qasm,9,9,0
system9.qasm,13,8,5



Plans where strict hotspot suffix-swap is structurally possible:


,circuit,plan_id,hotspot_chains,hotspot_lengths,max_weighted_degree,raw_strict_suffix_swap_attempts
1,system9.qasm,P1,"((2, 4, 6, 8),)","(4,)",71.0,6
3,system9.qasm,P11,"((0, 10),)","(2,)",65.0,4
4,system9.qasm,P12,"((2, 4, 6, 8),)","(4,)",71.0,6
5,system9.qasm,P2,"((0, 2, 4, 6, 8),)","(5,)",77.0,4
6,system9.qasm,P3,"((0, 2, 4, 6),)","(4,)",67.0,6
10,system9.qasm,P7,"((0, 2, 4, 6),)","(4,)",67.0,6
11,system9.qasm,P8,"((2, 4, 6, 8),)","(4,)",71.0,6
12,system9.qasm,P9,"((0, 2, 4, 6),)","(4,)",67.0,6
13,sym9_146.qasm,P0,"((0, 3, 5, 7, 8),)","(5,)",76.0,12
14,sym9_146.qasm,P1,"((2, 4, 6, 8),)","(4,)",71.0,12


In [23]:
from qiskit import QuantumCircuit, ClassicalRegister


def normalize_active_reset(qc: QuantumCircuit) -> QuantumCircuit:

    new_qc = QuantumCircuit(*qc.qregs, *qc.cregs)

    last_measure_clbit = {}
    clbit_owner = {}

    total_resets = sum(
        1
        for inst in qc.data
        if inst.operation.name == "reset"
    )

    aux_reg = None
    aux_next = 0

    for item in qc.data:

        instr = item.operation
        qargs = list(item.qubits)
        cargs = list(item.clbits)

        q_indices = [
            qc.find_bit(q).index
            for q in qargs
        ]

        # -----------------------------
        # Measurement
        # -----------------------------

        if instr.name == "measure":

            q = qargs[0]
            c = cargs[0]

            q_idx = qc.find_bit(q).index
            c_idx = qc.find_bit(c).index

            old_owner = clbit_owner.get(c_idx)

            if old_owner is not None:
                last_measure_clbit.pop(
                    old_owner,
                    None
                )

            new_qc.append(
                instr,
                qargs,
                cargs
            )

            new_cbit = new_qc.clbits[c_idx]

            last_measure_clbit[q_idx] = (
                new_cbit
            )

            clbit_owner[c_idx] = q_idx

            continue

        # -----------------------------
        # Reset
        # -----------------------------

        if instr.name == "reset":

            q = qargs[0]

            q_idx = qc.find_bit(q).index

            if q_idx in last_measure_clbit:

                cbit = (
                    last_measure_clbit[q_idx]
                )

            else:

                if aux_reg is None:

                    aux_reg = ClassicalRegister(
                        max(total_resets, 1),
                        "reset_aux"
                    )

                    new_qc.add_register(
                        aux_reg
                    )

                cbit = aux_reg[aux_next]

                aux_next += 1

                new_qc.measure(
                    q,
                    cbit
                )

            new_qc.x(q).c_if(
                cbit,
                1
            )

            last_measure_clbit.pop(
                q_idx,
                None
            )

            continue

        # -----------------------------
        # Other operation
        # -----------------------------

        new_qc.append(
            instr,
            qargs,
            cargs
        )

        for q_idx in q_indices:

            last_measure_clbit.pop(
                q_idx,
                None
            )

    return new_qc

In [24]:
# ------------------------------------------------------------
# Compile a FIXED reuse plan through GidNET's own pipeline
# ------------------------------------------------------------

def get_active_qubit_count(qc):

    active = set()

    for item in qc.data:

        for q in item.qubits:

            active.add(
                qc.find_bit(q).index
            )

    return len(active)


def compile_fixed_active_plan(
    circuit,
    active_chains
):
    """
    Compile exactly the supplied active reuse plan using
    GidNET's existing dynamic-circuit construction.

    GidNET's sequence-selection step is replaced ONLY for
    this instance; the rest of the compilation pipeline is
    unchanged.
    """

    full_chains = (
        expand_with_inactive_singletons(
            circuit,
            active_chains
        )
    )

    g = GidNET(
        copy.deepcopy(circuit)
    )

    # Force GidNET to use exactly this plan.
    g.compute_optimized_reuse_sequences = (
        lambda iterations:
            copy.deepcopy(full_chains)
    )

    dynamic_circuit = (
        g.compile_to_dynamic_circuit(
            iterations=1,
            draw=False
        )
    )

    return dynamic_circuit, g

In [25]:
# ------------------------------------------------------------
# Select a structurally applicable seed plan
# system9 / P1
# ------------------------------------------------------------

pilot_row = exp2_plans[
    (exp2_plans["circuit"] == "system9.qasm")
    &
    (exp2_plans["plan_id"] == "P1")
].iloc[0]

pilot_chains = parse_saved_structure(
    pilot_row["reuse_chains"]
)

pilot_chains = [
    tuple(chain)
    for chain in pilot_chains
]

pilot_circuit = circuit_cache[
    "system9.qasm"
]

print("Pilot seed:")
print(
    pilot_row["circuit"],
    pilot_row["plan_id"]
)

print("\nReuse chains:")
for chain in pilot_chains:
    print(" ", chain)

print(
    "\nSaved active width:",
    int(pilot_row["width"])
)

print(
    "Saved normalized logical depth:",
    pilot_row[
        "normalized_logical_depth"
    ]
)

print(
    "Saved max weighted degree:",
    pilot_row[
        "max_weighted_degree"
    ]
)

print(
    "Saved weighted-degree CV:",
    pilot_row[
        "weighted_degree_cv"
    ]
)

Pilot seed:
system9.qasm P1

Reuse chains:
  (0, 5, 7)
  (1,)
  (2, 4, 6, 8)
  (3,)
  (9,)
  (10,)
  (11,)

Saved active width: 7
Saved normalized logical depth: 176
Saved max weighted degree: 71.0
Saved weighted-degree CV: 0.4853635521727788


In [26]:
pilot_dynamic, pilot_gid = (
    compile_fixed_active_plan(
        pilot_circuit,
        pilot_chains
    )
)

pilot_dynamic_norm = (
    normalize_active_reset(
        pilot_dynamic
    )
)

pilot_active_width = (
    get_active_qubit_count(
        pilot_dynamic
    )
)

pilot_raw_depth = (
    pilot_dynamic.depth()
)

pilot_norm_depth = (
    pilot_dynamic_norm.depth()
)

print(
    "Recompiled active width:",
    pilot_active_width
)

print(
    "Recompiled raw logical depth:",
    pilot_raw_depth
)

print(
    "Recompiled normalized logical depth:",
    pilot_norm_depth
)

print(
    "\nSaved normalized logical depth:",
    int(
        pilot_row[
            "normalized_logical_depth"
        ]
    )
)

print(
    "Normalized-depth exact match:",
    pilot_norm_depth
    ==
    int(
        pilot_row[
            "normalized_logical_depth"
        ]
    )
)

assert (
    pilot_active_width
    ==
    int(pilot_row["width"])
)

assert (
    pilot_norm_depth
    ==
    int(
        pilot_row[
            "normalized_logical_depth"
        ]
    )
)

print(
    "\n✓ Fixed-plan compilation reproduces "
    "the Experiment-1/2 seed plan."
)

Recompiled active width: 7
Recompiled raw logical depth: 172
Recompiled normalized logical depth: 176

Saved normalized logical depth: 176
Normalized-depth exact match: True

✓ Fixed-plan compilation reproduces the Experiment-1/2 seed plan.


In [27]:
pilot_metrics = (
    compute_quotient_load_metrics(
        pilot_circuit,
        pilot_chains
    )
)

pilot_hotspots = (
    pilot_metrics[
        "hotspot_indices"
    ]
)

print(
    "Pilot hotspot indices:",
    pilot_hotspots
)

for idx in pilot_hotspots:

    print(
        f"  Chain {idx}: "
        f"{pilot_chains[idx]}"
        f" | weighted degree="
        f"{pilot_metrics['weighted_degree'][idx]}"
    )


pilot_neighborhood = (
    generate_hotspot_suffix_swap_neighbors(
        pilot_circuit,
        pilot_chains,
        pilot_hotspots
    )
)

print(
    "\nRaw suffix-swap attempts:",
    pilot_neighborhood[
        "generated_raw"
    ]
)

print(
    "DAG-legal unique neighbors:",
    len(
        pilot_neighborhood[
            "legal_neighbors"
        ]
    )
)

print(
    "Structural rejections:",
    pilot_neighborhood[
        "rejection_reasons"
    ]
)

Pilot hotspot indices: [2]
  Chain 2: (2, 4, 6, 8) | weighted degree=71.0

Raw suffix-swap attempts: 6
DAG-legal unique neighbors: 1
Structural rejections: {'boundary_not_in_candidate_matrix': 5}


In [28]:
# ------------------------------------------------------------
# Full pre-routing evaluation of legal P1 neighbors
# ------------------------------------------------------------

pilot_candidate_rows = []

compile_rejections = []

seed_max = (
    pilot_metrics[
        "max_weighted_degree"
    ]
)

seed_cv = (
    pilot_metrics[
        "weighted_degree_cv"
    ]
)

seed_depth = pilot_norm_depth

seed_width = (
    int(pilot_row["width"])
)


for i, neighbor in enumerate(
    pilot_neighborhood[
        "legal_neighbors"
    ]
):

    candidate_id = f"N{i}"

    chains = neighbor[
        "candidate_chains"
    ]

    # ----------------------------------------
    # Actual GidNET dynamic-circuit compilation
    # ----------------------------------------

    try:

        candidate_dynamic, _ = (
            compile_fixed_active_plan(
                pilot_circuit,
                chains
            )
        )

        candidate_active_width = (
            get_active_qubit_count(
                candidate_dynamic
            )
        )

        candidate_norm = (
            normalize_active_reset(
                candidate_dynamic
            )
        )

        candidate_depth = (
            candidate_norm.depth()
        )

    except Exception as e:

        compile_rejections.append({
            "candidate_id":
                candidate_id,

            "reason":
                "dynamic_compile_failed",

            "error":
                repr(e),
        })

        continue

    # ----------------------------------------
    # Width must remain unchanged
    # ----------------------------------------

    if candidate_active_width != seed_width:

        compile_rejections.append({
            "candidate_id":
                candidate_id,

            "reason":
                "active_width_changed",

            "observed_width":
                candidate_active_width,
        })

        continue

    # ----------------------------------------
    # Experiment-2 structural metrics
    # ----------------------------------------

    metrics = (
        compute_quotient_load_metrics(
            pilot_circuit,
            chains
        )
    )

    # ----------------------------------------
    # Logical-depth guard
    # ----------------------------------------

    relative_depth_change = (
        (
            candidate_depth
            - seed_depth
        )
        / seed_depth
    )

    depth_guard_pass = (
        relative_depth_change
        <= LOGICAL_DEPTH_TOLERANCE
    )

    # ----------------------------------------
    # Hotspot-improvement criterion
    # ----------------------------------------

    hotspot_improved = (
        metrics[
            "max_weighted_degree"
        ] < seed_max
    ) or (
        np.isclose(
            metrics[
                "max_weighted_degree"
            ],
            seed_max
        )
        and
        metrics[
            "weighted_degree_cv"
        ] < seed_cv
    )

    pilot_candidate_rows.append({

        "candidate_id":
            candidate_id,

        "other_chain_idx":
            neighbor[
                "other_chain_idx"
            ],

        "cut_hotspot":
            neighbor[
                "cut_hotspot"
            ],

        "cut_other":
            neighbor[
                "cut_other"
            ],

        "candidate_chains":
            chains,

        "removed_boundaries":
            neighbor[
                "removed_boundaries"
            ],

        "added_boundaries":
            neighbor[
                "added_boundaries"
            ],

        "active_width":
            candidate_active_width,

        "normalized_logical_depth":
            candidate_depth,

        "relative_depth_change":
            relative_depth_change,

        "depth_guard_pass":
            depth_guard_pass,

        "max_weighted_degree":
            metrics[
                "max_weighted_degree"
            ],

        "weighted_degree_cv":
            metrics[
                "weighted_degree_cv"
            ],

        "delta_max_weighted_degree":
            metrics[
                "max_weighted_degree"
            ] - seed_max,

        "delta_weighted_degree_cv":
            metrics[
                "weighted_degree_cv"
            ] - seed_cv,

        "hotspot_improved":
            hotspot_improved,
    })


pilot_candidate_df = pd.DataFrame(
    pilot_candidate_rows
)

print(
    "Full compilation rejections:"
)

print(
    compile_rejections
)

print(
    "\nSuccessfully compiled candidates:",
    len(pilot_candidate_df)
)

Full compilation rejections:
[]

Successfully compiled candidates: 1


In [29]:
if len(pilot_candidate_df) > 0:

    pilot_candidate_df = (
        pilot_candidate_df
        .sort_values(
            [
                "max_weighted_degree",
                "weighted_degree_cv",
            ]
        )
        .reset_index(drop=True)
    )

    display(
        pilot_candidate_df[
            [
                "candidate_id",
                "active_width",
                "normalized_logical_depth",
                "relative_depth_change",
                "depth_guard_pass",
                "max_weighted_degree",
                "weighted_degree_cv",
                "delta_max_weighted_degree",
                "delta_weighted_degree_cv",
                "hotspot_improved",
            ]
        ]
    )

else:

    print(
        "No candidate survived full compilation."
    )

,candidate_id,active_width,normalized_logical_depth,relative_depth_change,depth_guard_pass,max_weighted_degree,weighted_degree_cv,delta_max_weighted_degree,delta_weighted_degree_cv,hotspot_improved
0,N0,7,164,-0.068182,True,63.0,0.458161,-8.0,-0.027203,True


In [30]:
if len(pilot_candidate_df) > 0:

    pilot_eligible = (
        pilot_candidate_df[
            (
                pilot_candidate_df[
                    "hotspot_improved"
                ]
            )
            &
            (
                pilot_candidate_df[
                    "depth_guard_pass"
                ]
            )
        ]
        .copy()
    )

else:

    pilot_eligible = pd.DataFrame()


print(
    "Eligible targeted repairs:",
    len(pilot_eligible)
)


if len(pilot_eligible) > 0:

    selected = (
        pilot_eligible
        .sort_values(
            [
                "max_weighted_degree",
                "weighted_degree_cv",
            ]
        )
        .iloc[0]
    )

    print(
        "\nSELECTED TARGETED REPAIR:"
    )

    print(
        "Candidate:",
        selected[
            "candidate_id"
        ]
    )

    print(
        "max_weighted_degree:",
        seed_max,
        "->",
        selected[
            "max_weighted_degree"
        ]
    )

    print(
        "weighted_degree_cv:",
        seed_cv,
        "->",
        selected[
            "weighted_degree_cv"
        ]
    )

    print(
        "normalized logical depth:",
        seed_depth,
        "->",
        selected[
            "normalized_logical_depth"
        ]
    )

    print(
        "\nRemoved boundaries:",
        selected[
            "removed_boundaries"
        ]
    )

    print(
        "Added boundaries:",
        selected[
            "added_boundaries"
        ]
    )

    print(
        "\nCandidate chains:"
    )

    for chain in selected[
        "candidate_chains"
    ]:

        print(" ", chain)

else:

    print(
        "\nP1 has no eligible targeted repair "
        "under the current operator + 5% depth guard."
    )

Eligible targeted repairs: 1

SELECTED TARGETED REPAIR:
Candidate: N0
max_weighted_degree: 71.0 -> 63.0
weighted_degree_cv: 0.4853635521727788 -> 0.4581605735013775
normalized logical depth: 176 -> 164

Removed boundaries: [(0, 5), (2, 4)]
Added boundaries: [(0, 4), (2, 5)]

Candidate chains:
  (0, 4, 6, 8)
  (1,)
  (2, 5, 7)
  (3,)
  (9,)
  (10,)
  (11,)


In [31]:
# ------------------------------------------------------------
# Recover ORIGINAL P1 routing results from Experiment 1
# ------------------------------------------------------------

p1_original_raw = (
    exp1_raw[
        (exp1_raw["circuit"] == "system9.qasm")
        &
        (exp1_raw["plan_id"] == "P1")
    ]
    .sort_values("routing_seed")
    .copy()
)

display(
    p1_original_raw[
        [
            "routing_seed",
            "swap_count",
            "routed_depth",
            "cx_count",
        ]
    ]
)

print(
    "Routing seeds:",
    p1_original_raw[
        "routing_seed"
    ].tolist()
)

assert (
    p1_original_raw[
        "routing_seed"
    ].tolist()
    ==
    ROUTING_SEEDS
)

assert len(p1_original_raw) == 5

print(
    "\n✓ Original P1 has the exact paired "
    "routing seeds required for Experiment 3."
)

,routing_seed,swap_count,routed_depth,cx_count
5,42,96,310,150
6,43,93,315,150
7,44,96,307,150
8,45,94,287,150
9,46,92,312,150


Routing seeds: [42, 43, 44, 45, 46]

✓ Original P1 has the exact paired routing seeds required for Experiment 3.


In [32]:
# ------------------------------------------------------------
# Exact Experiment-1 physical-routing setup
# ------------------------------------------------------------

from qiskit import transpile
from qiskit.transpiler import CouplingMap

TARGET_COUPLING = (
    CouplingMap.from_heavy_hex(7)
)

print(
    "Heavy-Hex physical qubits:",
    TARGET_COUPLING.size()
)

Heavy-Hex physical qubits: 115


In [33]:
# ------------------------------------------------------------
# Physical evaluation of the selected targeted repair N0
# ------------------------------------------------------------

selected_chains = (
    selected[
        "candidate_chains"
    ]
)

# Recompile once from the selected fixed reuse plan
selected_dynamic, _ = (
    compile_fixed_active_plan(
        pilot_circuit,
        selected_chains
    )
)

selected_dynamic_norm = (
    normalize_active_reset(
        selected_dynamic
    )
)

print(
    "Selected candidate normalized logical depth:",
    selected_dynamic_norm.depth()
)

candidate_routing_rows = []

for routing_seed in ROUTING_SEEDS:

    routed = transpile(
        selected_dynamic_norm,
        coupling_map=TARGET_COUPLING,
        routing_method="sabre",
        layout_method="sabre",
        optimization_level=1,
        seed_transpiler=routing_seed,
    )

    candidate_routing_rows.append({
        "routing_seed":
            routing_seed,

        "swap_count":
            routed.count_ops().get(
                "swap",
                0
            ),

        "routed_depth":
            routed.depth(),

        "cx_count":
            routed.count_ops().get(
                "cx",
                0
            ),
    })


p1_repaired_raw = pd.DataFrame(
    candidate_routing_rows
)

display(p1_repaired_raw)

Selected candidate normalized logical depth: 164


2026-09-28 11:19:14,174 - INFO - Pass: UnrollCustomDefinitions - 0.00000 (ms)
2026-09-28 11:19:14,187 - INFO - Begin BasisTranslator from source basis {('u3', 1)} to target basis {'measure', 'delay', 'ry', 'u', 'cx', 'reset', 'barrier', 'z', 'snapshot', 'rx', 'x', 'p', 'rz'}.
2026-09-28 11:19:14,189 - INFO - Basis translation path search completed in 0.000s.
2026-09-28 11:19:14,189 - INFO - Basis translation paths composed in 0.000s.
2026-09-28 11:19:14,195 - INFO - Basis translation instructions replaced in 0.000s.
2026-09-28 11:19:14,198 - INFO - Pass: BasisTranslator - 11.25383 (ms)
2026-09-28 11:19:14,264 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:19:14,264 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:19:14,268 - INFO - Pass: HighLevelSynthesis - 2.01058 (ms)
2026-09-28 11:19:14,268 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:19:14,268 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:19:14,273 - INFO - Pass: BarrierBeforeFina

,routing_seed,swap_count,routed_depth,cx_count
0,42,85,288,150
1,43,85,283,150
2,44,85,284,150
3,45,86,298,150
4,46,85,287,150


In [34]:
# ------------------------------------------------------------
# Paired physical comparison: original P1 vs repaired N0
# ------------------------------------------------------------

paired = (
    p1_original_raw[
        [
            "routing_seed",
            "swap_count",
            "routed_depth",
            "cx_count",
        ]
    ]
    .rename(
        columns={
            "swap_count":
                "swaps_before",

            "routed_depth":
                "routed_depth_before",

            "cx_count":
                "cx_before",
        }
    )
    .merge(
        p1_repaired_raw.rename(
            columns={
                "swap_count":
                    "swaps_after",

                "routed_depth":
                    "routed_depth_after",

                "cx_count":
                    "cx_after",
            }
        ),
        on="routing_seed",
        how="inner",
    )
)

paired[
    "delta_swaps"
] = (
    paired["swaps_after"]
    -
    paired["swaps_before"]
)

paired[
    "delta_routed_depth"
] = (
    paired["routed_depth_after"]
    -
    paired["routed_depth_before"]
)

paired[
    "swap_improvement"
] = (
    paired["swaps_before"]
    -
    paired["swaps_after"]
)

paired[
    "depth_improvement"
] = (
    paired["routed_depth_before"]
    -
    paired["routed_depth_after"]
)

display(paired)

,routing_seed,swaps_before,routed_depth_before,cx_before,swaps_after,routed_depth_after,cx_after,delta_swaps,delta_routed_depth,swap_improvement,depth_improvement
0,42,96,310,150,85,288,150,-11,-22,11,22
1,43,93,315,150,85,283,150,-8,-32,8,32
2,44,96,307,150,85,284,150,-11,-23,11,23
3,45,94,287,150,86,298,150,-8,11,8,-11
4,46,92,312,150,85,287,150,-7,-25,7,25


In [35]:
# ------------------------------------------------------------
# Pilot intervention summary
# ------------------------------------------------------------

mean_swaps_before = (
    paired[
        "swaps_before"
    ].mean()
)

mean_swaps_after = (
    paired[
        "swaps_after"
    ].mean()
)

mean_depth_before = (
    paired[
        "routed_depth_before"
    ].mean()
)

mean_depth_after = (
    paired[
        "routed_depth_after"
    ].mean()
)

swap_improvement_pct = (
    100
    *
    (
        mean_swaps_before
        -
        mean_swaps_after
    )
    /
    mean_swaps_before
)

depth_improvement_pct = (
    100
    *
    (
        mean_depth_before
        -
        mean_depth_after
    )
    /
    mean_depth_before
)


print("========== Experiment 3A Pilot ==========")

print(
    "\nStructural intervention:"
)

print(
    "max_weighted_degree:",
    seed_max,
    "->",
    selected[
        "max_weighted_degree"
    ]
)

print(
    "weighted_degree_cv:",
    seed_cv,
    "->",
    selected[
        "weighted_degree_cv"
    ]
)

print(
    "normalized logical depth:",
    seed_depth,
    "->",
    selected[
        "normalized_logical_depth"
    ]
)

print(
    "\nPhysical routing:"
)

print(
    f"Mean SWAPs: "
    f"{mean_swaps_before:.2f}"
    f" -> "
    f"{mean_swaps_after:.2f}"
)

print(
    f"SWAP improvement: "
    f"{swap_improvement_pct:.2f}%"
)

print(
    f"\nMean routed depth: "
    f"{mean_depth_before:.2f}"
    f" -> "
    f"{mean_depth_after:.2f}"
)

print(
    f"Routed-depth improvement: "
    f"{depth_improvement_pct:.2f}%"
)

print(
    "\nRouting seeds with fewer SWAPs:",
    int(
        (
            paired[
                "delta_swaps"
            ] < 0
        ).sum()
    ),
    "/",
    len(paired)
)

print(
    "Routing seeds with lower routed depth:",
    int(
        (
            paired[
                "delta_routed_depth"
            ] < 0
        ).sum()
    ),
    "/",
    len(paired)
)

========== Experiment 3A Pilot ==========

Structural intervention:
max_weighted_degree: 71.0 -> 63.0
weighted_degree_cv: 0.4853635521727788 -> 0.4581605735013775
normalized logical depth: 176 -> 164

Physical routing:
Mean SWAPs: 94.20 -> 85.20
SWAP improvement: 9.55%

Mean routed depth: 306.20 -> 288.00
Routed-depth improvement: 5.94%

Routing seeds with fewer SWAPs: 5 / 5
Routing seeds with lower routed depth: 4 / 5


In [36]:
# ------------------------------------------------------------
# Experiment 3A — structural screening across ALL seed plans
# NO physical routing in this cell
# ------------------------------------------------------------

all_candidate_rows = []
seed_screening_rows = []

for row_idx, row in exp2_plans.iterrows():

    circuit_name = row["circuit"]
    plan_id = row["plan_id"]

    qc = circuit_cache[circuit_name]

    seed_chains = [
        tuple(chain)
        for chain in parse_saved_structure(
            row["reuse_chains"]
        )
    ]

    seed_width = int(row["width"])

    seed_depth = int(
        row["normalized_logical_depth"]
    )

    # --------------------------------------------------------
    # Seed metrics
    # --------------------------------------------------------

    seed_metrics = compute_quotient_load_metrics(
        qc,
        seed_chains
    )

    seed_max = seed_metrics[
        "max_weighted_degree"
    ]

    seed_cv = seed_metrics[
        "weighted_degree_cv"
    ]

    hotspot_indices = seed_metrics[
        "hotspot_indices"
    ]

    hotspot_chains = tuple(
        seed_chains[i]
        for i in hotspot_indices
    )

    # --------------------------------------------------------
    # Generate DAG-legal strict suffix-swap neighborhood
    # --------------------------------------------------------

    neighborhood = (
        generate_hotspot_suffix_swap_neighbors(
            qc,
            seed_chains,
            hotspot_indices
        )
    )

    raw_attempts = neighborhood[
        "generated_raw"
    ]

    legal_neighbors = neighborhood[
        "legal_neighbors"
    ]

    # --------------------------------------------------------
    # Full GidNET compilation + structural metrics
    # --------------------------------------------------------

    local_candidate_rows = []

    compile_failures = 0
    width_failures = 0

    for local_idx, neighbor in enumerate(
        legal_neighbors
    ):

        candidate_id = (
            f"{plan_id}_N{local_idx}"
        )

        candidate_chains = neighbor[
            "candidate_chains"
        ]

        # ----------------------------------------
        # Real fixed-plan GidNET compilation
        # ----------------------------------------

        try:

            candidate_dynamic, _ = (
                compile_fixed_active_plan(
                    qc,
                    candidate_chains
                )
            )

            candidate_width = (
                get_active_qubit_count(
                    candidate_dynamic
                )
            )

            candidate_norm = (
                normalize_active_reset(
                    candidate_dynamic
                )
            )

            candidate_depth = (
                candidate_norm.depth()
            )

        except Exception as e:

            compile_failures += 1

            all_candidate_rows.append({
                "circuit":
                    circuit_name,

                "seed_plan_id":
                    plan_id,

                "candidate_plan_id":
                    candidate_id,

                "repair_operator":
                    "two_chain_suffix_swap",

                "legal":
                    False,

                "failure_reason":
                    "dynamic_compile_failed",

                "error":
                    repr(e),
            })

            continue

        # ----------------------------------------
        # Active width must stay fixed
        # ----------------------------------------

        if candidate_width != seed_width:

            width_failures += 1

            all_candidate_rows.append({
                "circuit":
                    circuit_name,

                "seed_plan_id":
                    plan_id,

                "candidate_plan_id":
                    candidate_id,

                "repair_operator":
                    "two_chain_suffix_swap",

                "legal":
                    False,

                "failure_reason":
                    "active_width_changed",

                "width_before":
                    seed_width,

                "width_after":
                    candidate_width,
            })

            continue

        # ----------------------------------------
        # Quotient metrics
        # ----------------------------------------

        candidate_metrics = (
            compute_quotient_load_metrics(
                qc,
                candidate_chains
            )
        )

        candidate_max = (
            candidate_metrics[
                "max_weighted_degree"
            ]
        )

        candidate_cv = (
            candidate_metrics[
                "weighted_degree_cv"
            ]
        )

        delta_max = (
            candidate_max - seed_max
        )

        delta_cv = (
            candidate_cv - seed_cv
        )

        # ----------------------------------------
        # Depth guard
        # ----------------------------------------

        relative_depth_change = (
            candidate_depth - seed_depth
        ) / seed_depth

        depth_guard_pass = (
            relative_depth_change
            <= LOGICAL_DEPTH_TOLERANCE
        )

        # ----------------------------------------
        # Target structural criterion
        # ----------------------------------------

        hotspot_improved = (
            candidate_max < seed_max
        ) or (
            np.isclose(
                candidate_max,
                seed_max
            )
            and
            candidate_cv < seed_cv
        )

        candidate_record = {

            "circuit":
                circuit_name,

            "seed_plan_id":
                plan_id,

            "candidate_plan_id":
                candidate_id,

            "repair_operator":
                "two_chain_suffix_swap",

            "plan_signature_before":
                canonical_plan_signature(
                    seed_chains
                ),

            "plan_signature_after":
                canonical_plan_signature(
                    candidate_chains
                ),

            "candidate_chains":
                candidate_chains,

            "hotspot_chain_indices":
                tuple(hotspot_indices),

            "hotspot_chains":
                hotspot_chains,

            "other_chain_idx":
                neighbor[
                    "other_chain_idx"
                ],

            "cut_hotspot":
                neighbor[
                    "cut_hotspot"
                ],

            "cut_other":
                neighbor[
                    "cut_other"
                ],

            "removed_boundaries":
                neighbor[
                    "removed_boundaries"
                ],

            "added_boundaries":
                neighbor[
                    "added_boundaries"
                ],

            "width_before":
                seed_width,

            "width_after":
                candidate_width,

            "logical_depth_before":
                seed_depth,

            "logical_depth_after":
                candidate_depth,

            "relative_depth_change":
                relative_depth_change,

            "depth_guard_pass":
                depth_guard_pass,

            "max_weighted_degree_before":
                seed_max,

            "max_weighted_degree_after":
                candidate_max,

            "weighted_degree_cv_before":
                seed_cv,

            "weighted_degree_cv_after":
                candidate_cv,

            "delta_max_weighted_degree":
                delta_max,

            "delta_weighted_degree_cv":
                delta_cv,

            "hotspot_improved":
                hotspot_improved,

            "legal":
                True,

            "failure_reason":
                None,

            "selected_by_targeted_method":
                False,
        }

        local_candidate_rows.append(
            candidate_record
        )

        all_candidate_rows.append(
            candidate_record
        )

    # --------------------------------------------------------
    # Select best eligible candidate lexicographically
    # --------------------------------------------------------

    local_df = pd.DataFrame(
        local_candidate_rows
    )

    selected_candidate_id = None

    eligible_count = 0

    if len(local_df) > 0:

        eligible_df = local_df[
            local_df["hotspot_improved"]
            &
            local_df["depth_guard_pass"]
        ].copy()

        eligible_count = len(
            eligible_df
        )

        if eligible_count > 0:

            eligible_df = (
                eligible_df
                .sort_values(
                    [
                        "max_weighted_degree_after",
                        "weighted_degree_cv_after",
                    ]
                )
            )

            selected_candidate_id = (
                eligible_df.iloc[0][
                    "candidate_plan_id"
                ]
            )

            # Mark selected row in master list
            for record in all_candidate_rows[::-1]:

                if (
                    record.get("circuit")
                    == circuit_name
                    and
                    record.get("seed_plan_id")
                    == plan_id
                    and
                    record.get("candidate_plan_id")
                    == selected_candidate_id
                ):

                    record[
                        "selected_by_targeted_method"
                    ] = True

                    break

    # --------------------------------------------------------
    # Seed-level screening summary
    # --------------------------------------------------------

    seed_screening_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            plan_id,

        "width":
            seed_width,

        "hotspot_chains":
            hotspot_chains,

        "max_weighted_degree":
            seed_max,

        "weighted_degree_cv":
            seed_cv,

        "normalized_logical_depth":
            seed_depth,

        "raw_suffix_swap_attempts":
            raw_attempts,

        "dag_legal_neighbors":
            len(legal_neighbors),

        "compile_failures":
            compile_failures,

        "width_failures":
            width_failures,

        "fully_compiled_neighbors":
            len(local_candidate_rows),

        "eligible_targeted_neighbors":
            eligible_count,

        "eligible_seed":
            selected_candidate_id
            is not None,

        "selected_candidate_id":
            selected_candidate_id,
    })


experiment3_candidates = pd.DataFrame(
    all_candidate_rows
)

experiment3_screening = pd.DataFrame(
    seed_screening_rows
)

print("Screening complete.")

Screening complete.


In [37]:
# ------------------------------------------------------------
# Experiment 3A screening summary
# ------------------------------------------------------------

print("Total seed plans:")
print(len(experiment3_screening))

print(
    "\nSeeds with >=1 raw strict suffix swap:"
)

print(
    int(
        (
            experiment3_screening[
                "raw_suffix_swap_attempts"
            ] > 0
        ).sum()
    )
)

print(
    "\nSeeds with >=1 DAG-legal neighbor:"
)

print(
    int(
        (
            experiment3_screening[
                "dag_legal_neighbors"
            ] > 0
        ).sum()
    )
)

print(
    "\nSeeds with >=1 fully compiled neighbor:"
)

print(
    int(
        (
            experiment3_screening[
                "fully_compiled_neighbors"
            ] > 0
        ).sum()
    )
)

print(
    "\nELIGIBLE SEEDS "
    "(hotspot improvement + 5% depth guard):"
)

print(
    int(
        experiment3_screening[
            "eligible_seed"
        ].sum()
    )
)


summary_by_circuit = (
    experiment3_screening
    .groupby("circuit")
    .agg(
        total_seeds=(
            "seed_plan_id",
            "size"
        ),

        structurally_applicable=(
            "raw_suffix_swap_attempts",
            lambda x:
                int((x > 0).sum())
        ),

        has_legal_neighbor=(
            "dag_legal_neighbors",
            lambda x:
                int((x > 0).sum())
        ),

        eligible_seeds=(
            "eligible_seed",
            "sum"
        ),
    )
)

display(summary_by_circuit)

Total seed plans:
50

Seeds with >=1 raw strict suffix swap:
41

Seeds with >=1 DAG-legal neighbor:
17

Seeds with >=1 fully compiled neighbor:
17

ELIGIBLE SEEDS (hotspot improvement + 5% depth guard):
12


,total_seeds,structurally_applicable,has_legal_neighbor,eligible_seeds
circuit,,,,
alu-bdd_288.qasm,6,2,0,0
mini_alu_305.qasm,16,16,11,8
rd73_140.qasm,6,6,2,1
sym9_146.qasm,9,9,2,1
system9.qasm,13,8,2,2


In [38]:
selected_repairs = (
    experiment3_candidates[
        experiment3_candidates[
            "selected_by_targeted_method"
        ] == True
    ]
    .copy()
)

if len(selected_repairs) > 0:

    display(
        selected_repairs[
            [
                "circuit",
                "seed_plan_id",
                "candidate_plan_id",
                "max_weighted_degree_before",
                "max_weighted_degree_after",
                "delta_max_weighted_degree",
                "weighted_degree_cv_before",
                "weighted_degree_cv_after",
                "logical_depth_before",
                "logical_depth_after",
                "relative_depth_change",
            ]
        ]
        .sort_values(
            [
                "circuit",
                "seed_plan_id",
            ]
        )
    )

print(
    "Number of selected targeted repairs:",
    len(selected_repairs)
)

,circuit,seed_plan_id,candidate_plan_id,max_weighted_degree_before,max_weighted_degree_after,delta_max_weighted_degree,weighted_degree_cv_before,weighted_degree_cv_after,logical_depth_before,logical_depth_after,relative_depth_change
7,mini_alu_305.qasm,P1,P1_N1,44.0,40.0,-4.0,0.450820,0.425411,103,103,0.000000
9,mini_alu_305.qasm,P11,P11_N1,43.0,39.0,-4.0,0.431317,0.407180,102,102,0.000000
10,mini_alu_305.qasm,P12,P12_N0,44.0,43.0,-1.0,0.372797,0.364563,103,102,-0.009709
11,mini_alu_305.qasm,P13,P13_N0,54.0,53.0,-1.0,0.540172,0.529290,98,97,-0.010204
14,mini_alu_305.qasm,P3,P3_N1,44.0,40.0,-4.0,0.438300,0.412120,103,103,0.000000
15,mini_alu_305.qasm,P4,P4_N0,40.0,39.0,-1.0,0.342375,0.337912,95,95,0.000000
17,mini_alu_305.qasm,P7,P7_N0,40.0,39.0,-1.0,0.412120,0.407180,103,102,-0.009709
20,mini_alu_305.qasm,P9,P9_N0,54.0,53.0,-1.0,0.508322,0.493677,98,98,0.000000
4,rd73_140.qasm,P0,P0_N0,56.0,50.0,-6.0,0.284264,0.204878,139,139,0.000000
3,sym9_146.qasm,P5,P5_N0,82.0,76.0,-6.0,0.275788,0.212704,197,197,0.000000


Number of selected targeted repairs: 12


In [39]:
selected_repairs = (
    experiment3_candidates[
        experiment3_candidates[
            "selected_by_targeted_method"
        ] == True
    ]
    .copy()
)

if len(selected_repairs) > 0:

    display(
        selected_repairs[
            [
                "circuit",
                "seed_plan_id",
                "candidate_plan_id",
                "max_weighted_degree_before",
                "max_weighted_degree_after",
                "delta_max_weighted_degree",
                "weighted_degree_cv_before",
                "weighted_degree_cv_after",
                "logical_depth_before",
                "logical_depth_after",
                "relative_depth_change",
            ]
        ]
        .sort_values(
            [
                "circuit",
                "seed_plan_id",
            ]
        )
    )

print(
    "Number of selected targeted repairs:",
    len(selected_repairs)
)

,circuit,seed_plan_id,candidate_plan_id,max_weighted_degree_before,max_weighted_degree_after,delta_max_weighted_degree,weighted_degree_cv_before,weighted_degree_cv_after,logical_depth_before,logical_depth_after,relative_depth_change
7,mini_alu_305.qasm,P1,P1_N1,44.0,40.0,-4.0,0.450820,0.425411,103,103,0.000000
9,mini_alu_305.qasm,P11,P11_N1,43.0,39.0,-4.0,0.431317,0.407180,102,102,0.000000
10,mini_alu_305.qasm,P12,P12_N0,44.0,43.0,-1.0,0.372797,0.364563,103,102,-0.009709
11,mini_alu_305.qasm,P13,P13_N0,54.0,53.0,-1.0,0.540172,0.529290,98,97,-0.010204
14,mini_alu_305.qasm,P3,P3_N1,44.0,40.0,-4.0,0.438300,0.412120,103,103,0.000000
15,mini_alu_305.qasm,P4,P4_N0,40.0,39.0,-1.0,0.342375,0.337912,95,95,0.000000
17,mini_alu_305.qasm,P7,P7_N0,40.0,39.0,-1.0,0.412120,0.407180,103,102,-0.009709
20,mini_alu_305.qasm,P9,P9_N0,54.0,53.0,-1.0,0.508322,0.493677,98,98,0.000000
4,rd73_140.qasm,P0,P0_N0,56.0,50.0,-6.0,0.284264,0.204878,139,139,0.000000
3,sym9_146.qasm,P5,P5_N0,82.0,76.0,-6.0,0.275788,0.212704,197,197,0.000000


Number of selected targeted repairs: 12


In [43]:
# ------------------------------------------------------------
# Experiment 3A — Physical evaluation of all selected repairs
# 12 candidates × 5 paired SABRE seeds
# ------------------------------------------------------------

experiment3_raw_rows = []
routing_failures = []

for _, repair in selected_repairs.iterrows():

    circuit_name = repair["circuit"]
    seed_plan_id = repair["seed_plan_id"]
    candidate_plan_id = repair["candidate_plan_id"]

    print(
        f"\nRouting {circuit_name} | "
        f"{seed_plan_id} -> {candidate_plan_id}"
    )

    qc = circuit_cache[circuit_name]

    # Candidate chains
    candidate_chains_raw = repair[
        "candidate_chains"
    ]

    if isinstance(candidate_chains_raw, str):
        candidate_chains_raw = ast.literal_eval(
            candidate_chains_raw
        )

    candidate_chains = [
        tuple(chain)
        for chain in candidate_chains_raw
    ]

    # --------------------------------------------------------
    # Compile repaired reuse plan
    # --------------------------------------------------------

    try:

        candidate_dynamic, _ = (
            compile_fixed_active_plan(
                qc,
                candidate_chains
            )
        )

        candidate_norm = (
            normalize_active_reset(
                candidate_dynamic
            )
        )

    except Exception as e:

        routing_failures.append({
            "circuit": circuit_name,
            "seed_plan_id": seed_plan_id,
            "candidate_plan_id": candidate_plan_id,
            "stage": "fixed_plan_compile",
            "error": repr(e),
        })

        print("  COMPILE FAILED:", repr(e))
        continue

    # Sanity check against structural-screening result
    expected_depth = int(
        repair["logical_depth_after"]
    )

    observed_depth = (
        candidate_norm.depth()
    )

    assert observed_depth == expected_depth, (
        circuit_name,
        seed_plan_id,
        candidate_plan_id,
        observed_depth,
        expected_depth,
    )

    # --------------------------------------------------------
    # Original physical results from Experiment 1
    # --------------------------------------------------------

    original_raw = (
        exp1_raw[
            (exp1_raw["circuit"] == circuit_name)
            &
            (exp1_raw["plan_id"] == seed_plan_id)
        ]
        .sort_values("routing_seed")
        .copy()
    )

    assert (
        original_raw["routing_seed"].tolist()
        ==
        ROUTING_SEEDS
    )

    original_by_seed = (
        original_raw
        .set_index("routing_seed")
    )

    # --------------------------------------------------------
    # Paired physical routing
    # --------------------------------------------------------

    for routing_seed in ROUTING_SEEDS:

        try:

            routed = transpile(
                candidate_norm,
                coupling_map=TARGET_COUPLING,
                routing_method="sabre",
                layout_method="sabre",
                optimization_level=1,
                seed_transpiler=routing_seed,
            )

        except Exception as e:

            routing_failures.append({
                "circuit": circuit_name,
                "seed_plan_id": seed_plan_id,
                "candidate_plan_id": candidate_plan_id,
                "routing_seed": routing_seed,
                "stage": "physical_routing",
                "error": repr(e),
            })

            print(
                f"  seed {routing_seed}: FAILED"
            )

            continue

        before = original_by_seed.loc[
            routing_seed
        ]

        swaps_before = int(
            before["swap_count"]
        )

        depth_before = int(
            before["routed_depth"]
        )

        cx_before = int(
            before["cx_count"]
        )

        swaps_after = int(
            routed.count_ops().get(
                "swap",
                0
            )
        )

        depth_after = int(
            routed.depth()
        )

        cx_after = int(
            routed.count_ops().get(
                "cx",
                0
            )
        )

        experiment3_raw_rows.append({

            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "candidate_plan_id":
                candidate_plan_id,

            "routing_seed":
                routing_seed,

            "width_before":
                int(repair["width_before"]),

            "width_after":
                int(repair["width_after"]),

            "logical_depth_before":
                int(
                    repair[
                        "logical_depth_before"
                    ]
                ),

            "logical_depth_after":
                int(
                    repair[
                        "logical_depth_after"
                    ]
                ),

            "relative_depth_change":
                float(
                    repair[
                        "relative_depth_change"
                    ]
                ),

            "max_weighted_degree_before":
                float(
                    repair[
                        "max_weighted_degree_before"
                    ]
                ),

            "max_weighted_degree_after":
                float(
                    repair[
                        "max_weighted_degree_after"
                    ]
                ),

            "weighted_degree_cv_before":
                float(
                    repair[
                        "weighted_degree_cv_before"
                    ]
                ),

            "weighted_degree_cv_after":
                float(
                    repair[
                        "weighted_degree_cv_after"
                    ]
                ),

            "swaps_before":
                swaps_before,

            "swaps_after":
                swaps_after,

            "delta_swaps":
                swaps_after
                - swaps_before,

            "routed_depth_before":
                depth_before,

            "routed_depth_after":
                depth_after,

            "delta_routed_depth":
                depth_after
                - depth_before,

            "cx_before":
                cx_before,

            "cx_after":
                cx_after,
        })

        print(
            f"  seed {routing_seed}: "
            f"SWAP {swaps_before}->{swaps_after}, "
            f"depth {depth_before}->{depth_after}"
        )


experiment3_raw_routing = pd.DataFrame(
    experiment3_raw_rows
)

print(
    "\nCompleted routing rows:",
    len(experiment3_raw_routing)
)

print(
    "Routing failures:",
    len(routing_failures)
)


Routing system9.qasm | P1 -> P1_N0


2026-09-28 11:33:10,595 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:33:10,595 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:10,595 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:33:10,595 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:33:10,600 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:33:10,602 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 11:33:10,634 - INFO - Pass: SabreLayout - 32.04894 (ms)
2026-09-28 11:33:10,636 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:33:10,636 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:10,636 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:33:10,645 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:33:10,645 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:10,645 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:10,645 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:10,645 - INFO - Pas

  seed 42: SWAP 96->85, depth 310->288
  seed 43: SWAP 93->85, depth 315->283


2026-09-28 11:33:10,899 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:33:10,899 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:10,909 - INFO - Pass: HighLevelSynthesis - 4.02761 (ms)
2026-09-28 11:33:10,909 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:33:10,911 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:10,911 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:10,911 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:10,911 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:10,928 - INFO - Pass: Optimize1qGatesDecomposition - 16.76726 (ms)
2026-09-28 11:33:10,928 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 11:33:10,928 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:33:10,936 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:10,936 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:10,936 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:10,940 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-

  seed 44: SWAP 96->85, depth 307->284
  seed 45: SWAP 94->86, depth 287->298


2026-09-28 11:33:11,176 - INFO - Pass: CXCancellation - 12.72750 (ms)
2026-09-28 11:33:11,178 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:33:11,178 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:11,178 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:11,182 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:11,182 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:11,182 - INFO - Total Transpile Time - 100.14129 (ms)
2026-09-28 11:33:11,322 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:33:11,330 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:11,332 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:33:11,332 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:33:11,332 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:33:11,338 - INFO - Pass: BarrierBeforeFinalMeasurements - 5.60641 (ms)
2026-09-28 11:33:11,361 - INFO - Pass: SabreLayout - 23.29254 (ms)
2026-09-28 11:33:11,361 - INFO - Pass: C

... (45 outputs left out)

2026-09-28 11:33:17,215 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:33:17,217 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:33:17,218 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 11:33:17,233 - INFO - Pass: SabreLayout - 13.27801 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: CheckMap - 8.11791 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:17,241 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:17,250 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:17,250 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:17,258 - INFO - Pass: Optimize1qGatesDecomposition - 8.17394 (ms)
2026-09-28 11:33:17,258 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 11:33:17,258 - INFO - Pass

  seed 42: SWAP 46->46, depth 163->163
  seed 43: SWAP 46->47, depth 163->166
  seed 44: SWAP 46->45, depth 163->163


2026-09-28 11:33:17,481 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:33:17,489 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:33:17,489 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:33:17,489 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:33:17,494 - INFO - Pass: Depth - 5.48506 (ms)
2026-09-28 11:33:17,495 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:17,495 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:17,495 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:17,505 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 11:33:17,505 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 11:33:17,505 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:33:17,505 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:33:17,505 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:33:17,512 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:33:17,512 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-0

  seed 45: SWAP 48->47, depth 165->161
  seed 46: SWAP 48->45, depth 162->162

Completed routing rows: 60
Routing failures: 0


In [44]:
# ------------------------------------------------------------
# Intervention-level physical summaries
# ------------------------------------------------------------

intervention_rows = []

for (
    circuit_name,
    seed_plan_id,
    candidate_plan_id
), group in experiment3_raw_routing.groupby(
    [
        "circuit",
        "seed_plan_id",
        "candidate_plan_id",
    ]
):

    repair = selected_repairs[
        (
            selected_repairs["circuit"]
            == circuit_name
        )
        &
        (
            selected_repairs["seed_plan_id"]
            == seed_plan_id
        )
        &
        (
            selected_repairs[
                "candidate_plan_id"
            ]
            == candidate_plan_id
        )
    ].iloc[0]

    mean_swaps_before = (
        group["swaps_before"].mean()
    )

    mean_swaps_after = (
        group["swaps_after"].mean()
    )

    mean_depth_before = (
        group[
            "routed_depth_before"
        ].mean()
    )

    mean_depth_after = (
        group[
            "routed_depth_after"
        ].mean()
    )

    delta_swaps = (
        mean_swaps_after
        - mean_swaps_before
    )

    delta_depth = (
        mean_depth_after
        - mean_depth_before
    )

    swap_improvement_pct = (
        100
        *
        (
            mean_swaps_before
            - mean_swaps_after
        )
        /
        mean_swaps_before
    )

    depth_improvement_pct = (
        100
        *
        (
            mean_depth_before
            - mean_depth_after
        )
        /
        mean_depth_before
    )

    relative_logical_depth_change = float(
        repair["relative_depth_change"]
    )

    intervention_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            seed_plan_id,

        "candidate_plan_id":
            candidate_plan_id,

        "max_weighted_degree_before":
            float(
                repair[
                    "max_weighted_degree_before"
                ]
            ),

        "max_weighted_degree_after":
            float(
                repair[
                    "max_weighted_degree_after"
                ]
            ),

        "delta_max_weighted_degree":
            float(
                repair[
                    "delta_max_weighted_degree"
                ]
            ),

        "weighted_degree_cv_before":
            float(
                repair[
                    "weighted_degree_cv_before"
                ]
            ),

        "weighted_degree_cv_after":
            float(
                repair[
                    "weighted_degree_cv_after"
                ]
            ),

        "delta_weighted_degree_cv":
            float(
                repair[
                    "delta_weighted_degree_cv"
                ]
            ),

        "logical_depth_before":
            int(
                repair[
                    "logical_depth_before"
                ]
            ),

        "logical_depth_after":
            int(
                repair[
                    "logical_depth_after"
                ]
            ),

        "relative_logical_depth_change":
            relative_logical_depth_change,

        # Important controlled subset
        "depth_matched_5pct":
            abs(
                relative_logical_depth_change
            ) <= 0.05,

        "mean_swaps_before":
            mean_swaps_before,

        "mean_swaps_after":
            mean_swaps_after,

        "delta_swaps":
            delta_swaps,

        "swap_improvement_pct":
            swap_improvement_pct,

        "mean_routed_depth_before":
            mean_depth_before,

        "mean_routed_depth_after":
            mean_depth_after,

        "delta_routed_depth":
            delta_depth,

        "depth_improvement_pct":
            depth_improvement_pct,

        "swap_better_seeds":
            int(
                (
                    group["delta_swaps"]
                    < 0
                ).sum()
            ),

        "swap_equal_seeds":
            int(
                (
                    group["delta_swaps"]
                    == 0
                ).sum()
            ),

        "swap_worse_seeds":
            int(
                (
                    group["delta_swaps"]
                    > 0
                ).sum()
            ),

        "depth_better_seeds":
            int(
                (
                    group[
                        "delta_routed_depth"
                    ] < 0
                ).sum()
            ),

        "depth_equal_seeds":
            int(
                (
                    group[
                        "delta_routed_depth"
                    ] == 0
                ).sum()
            ),

        "depth_worse_seeds":
            int(
                (
                    group[
                        "delta_routed_depth"
                    ] > 0
                ).sum()
            ),

        "num_routing_trials":
            len(group),
    })


experiment3_interventions = pd.DataFrame(
    intervention_rows
)

display(
    experiment3_interventions[
        [
            "circuit",
            "seed_plan_id",
            "candidate_plan_id",

            "delta_max_weighted_degree",
            "delta_weighted_degree_cv",

            "logical_depth_before",
            "logical_depth_after",
            "depth_matched_5pct",

            "mean_swaps_before",
            "mean_swaps_after",
            "swap_improvement_pct",

            "mean_routed_depth_before",
            "mean_routed_depth_after",
            "depth_improvement_pct",

            "swap_better_seeds",
            "depth_better_seeds",
        ]
    ]
    .sort_values(
        [
            "circuit",
            "seed_plan_id",
        ]
    )
)

,circuit,seed_plan_id,candidate_plan_id,delta_max_weighted_degree,delta_weighted_degree_cv,logical_depth_before,logical_depth_after,depth_matched_5pct,mean_swaps_before,mean_swaps_after,swap_improvement_pct,mean_routed_depth_before,mean_routed_depth_after,depth_improvement_pct,swap_better_seeds,depth_better_seeds
0,mini_alu_305.qasm,P1,P1_N1,-4.0,-0.025408,103,103,True,43.0,43.0,0.000000,158.2,158.6,-0.252845,2,1
1,mini_alu_305.qasm,P11,P11_N1,-4.0,-0.024138,102,102,True,44.2,45.2,-2.262443,157.4,158.6,-0.762389,0,1
2,mini_alu_305.qasm,P12,P12_N0,-1.0,-0.008235,103,102,True,41.8,42.2,-0.956938,154.8,153.2,1.033592,2,4
3,mini_alu_305.qasm,P13,P13_N0,-1.0,-0.010882,98,97,True,45.4,44.4,2.202643,159.6,157.8,1.127820,2,3
4,mini_alu_305.qasm,P3,P3_N1,-4.0,-0.026179,103,103,True,44.4,45.4,-2.252252,157.4,157.2,0.127065,1,2
5,mini_alu_305.qasm,P4,P4_N0,-1.0,-0.004463,95,95,True,44.2,43.6,1.357466,156.0,155.8,0.128205,3,1
6,mini_alu_305.qasm,P7,P7_N0,-1.0,-0.004941,103,102,True,45.6,45.2,0.877193,157.0,158.6,-1.019108,3,0
7,mini_alu_305.qasm,P9,P9_N0,-1.0,-0.014644,98,98,True,46.8,46.0,1.709402,163.2,163.0,0.122549,3,1
8,rd73_140.qasm,P0,P0_N0,-6.0,-0.079387,139,139,True,60.4,52.2,13.576159,227.0,212.6,6.343612,5,5
9,sym9_146.qasm,P5,P5_N0,-6.0,-0.063085,197,197,True,90.0,86.6,3.777778,323.2,315.6,2.351485,4,5


In [45]:
# ------------------------------------------------------------
# Overall Experiment 3A intervention summary
# ------------------------------------------------------------

df = experiment3_interventions

print(
    "Total targeted interventions:",
    len(df)
)

print(
    "Depth-matched interventions (|Δlogical depth| <= 5%):",
    int(
        df["depth_matched_5pct"].sum()
    )
)

print(
    "\nInterventions with lower mean SWAP:",
    int(
        (
            df["delta_swaps"] < 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "Interventions with equal mean SWAP:",
    int(
        (
            df["delta_swaps"] == 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "Interventions with higher mean SWAP:",
    int(
        (
            df["delta_swaps"] > 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "\nInterventions with lower mean routed depth:",
    int(
        (
            df["delta_routed_depth"] < 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "Interventions with higher mean routed depth:",
    int(
        (
            df["delta_routed_depth"] > 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "\nMedian SWAP improvement (%):",
    df[
        "swap_improvement_pct"
    ].median()
)

print(
    "Mean SWAP improvement (%):",
    df[
        "swap_improvement_pct"
    ].mean()
)

print(
    "\nMedian routed-depth improvement (%):",
    df[
        "depth_improvement_pct"
    ].median()
)

print(
    "Mean routed-depth improvement (%):",
    df[
        "depth_improvement_pct"
    ].mean()
)

Total targeted interventions: 12
Depth-matched interventions (|Δlogical depth| <= 5%): 10

Interventions with lower mean SWAP: 8 / 12
Interventions with equal mean SWAP: 1 / 12
Interventions with higher mean SWAP: 3 / 12

Interventions with lower mean routed depth: 9 / 12
Interventions with higher mean routed depth: 3 / 12

Median SWAP improvement (%): 1.5334338863750614
Mean SWAP improvement (%): 2.485289428995599

Median routed-depth improvement (%): 0.5808984297356427
Mean routed-depth improvement (%): 1.558037104813707


In [46]:
# ------------------------------------------------------------
# Depth-controlled subset
# ------------------------------------------------------------

depth_matched = (
    experiment3_interventions[
        experiment3_interventions[
            "depth_matched_5pct"
        ]
    ]
    .copy()
)

print(
    "DEPTH-MATCHED SUBSET:",
    len(depth_matched)
)

print(
    "\nLower mean SWAP:",
    int(
        (
            depth_matched[
                "delta_swaps"
            ] < 0
        ).sum()
    ),
    "/",
    len(depth_matched)
)

print(
    "Lower mean routed depth:",
    int(
        (
            depth_matched[
                "delta_routed_depth"
            ] < 0
        ).sum()
    ),
    "/",
    len(depth_matched)
)

print(
    "\nMean SWAP improvement (%):",
    depth_matched[
        "swap_improvement_pct"
    ].mean()
)

print(
    "Median SWAP improvement (%):",
    depth_matched[
        "swap_improvement_pct"
    ].median()
)

print(
    "\nMean routed-depth improvement (%):",
    depth_matched[
        "depth_improvement_pct"
    ].mean()
)

print(
    "Median routed-depth improvement (%):",
    depth_matched[
        "depth_improvement_pct"
    ].median()
)

DEPTH-MATCHED SUBSET: 10

Lower mean SWAP: 6 / 10
Lower mean routed depth: 7 / 10

Mean SWAP improvement (%): 1.8029007154978198
Median SWAP improvement (%): 1.1173295229022784

Mean routed-depth improvement (%): 0.919998611513325
Median routed-depth improvement (%): 0.1276349656273435


In [47]:
# ------------------------------------------------------------
# Save Experiment 3A results so far
# ------------------------------------------------------------

EXP3_DIR = ROOT / "experiment3_results"
EXP3_DIR.mkdir(
    parents=True,
    exist_ok=True
)

experiment3_candidates.to_csv(
    EXP3_DIR / "experiment3_candidates.csv",
    index=False
)

experiment3_screening.to_csv(
    EXP3_DIR / "experiment3_screening.csv",
    index=False
)

experiment3_raw_routing.to_csv(
    EXP3_DIR / "experiment3_raw_routing.csv",
    index=False
)

experiment3_interventions.to_csv(
    EXP3_DIR / "experiment3_interventions.csv",
    index=False
)

print(
    "Saved to:",
    EXP3_DIR
)

Saved to: C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm\experiment3_results


In [48]:
# ------------------------------------------------------------
# Experiment 3A — Random-baseline feasibility
#
# For each targeted seed, inspect the SAME legal neighborhood.
# Random baseline uses:
#   - same suffix-swap operator
#   - same legality requirements
#   - same width
#   - same 5% logical-depth guard
#
# It does NOT require hotspot improvement.
# ------------------------------------------------------------

selected_keys = (
    selected_repairs[
        [
            "circuit",
            "seed_plan_id",
            "candidate_plan_id",
        ]
    ]
    .rename(
        columns={
            "candidate_plan_id":
                "targeted_candidate_id"
        }
    )
    .copy()
)


random_baseline_pool = (
    experiment3_candidates
    .merge(
        selected_keys,
        on=[
            "circuit",
            "seed_plan_id",
        ],
        how="inner"
    )
    .copy()
)


# Keep only candidates that:
# 1. are fully legal
# 2. passed the same logical-depth guard
random_baseline_pool = (
    random_baseline_pool[
        (
            random_baseline_pool["legal"]
            == True
        )
        &
        (
            random_baseline_pool[
                "depth_guard_pass"
            ] == True
        )
    ]
    .copy()
)


# Is this candidate the targeted choice?
random_baseline_pool[
    "is_targeted_candidate"
] = (
    random_baseline_pool[
        "candidate_plan_id"
    ]
    ==
    random_baseline_pool[
        "targeted_candidate_id"
    ]
)


pool_summary = (
    random_baseline_pool
    .groupby(
        [
            "circuit",
            "seed_plan_id",
        ]
    )
    .agg(
        num_legal_depth_guard_candidates=(
            "candidate_plan_id",
            "size"
        ),

        num_non_target_candidates=(
            "is_targeted_candidate",
            lambda x: int((~x).sum())
        ),

        targeted_candidate_id=(
            "targeted_candidate_id",
            "first"
        ),
    )
    .reset_index()
)


pool_summary[
    "selection_comparison_possible"
] = (
    pool_summary[
        "num_legal_depth_guard_candidates"
    ] >= 2
)


display(pool_summary)


print(
    "\nTargeted interventions:",
    len(selected_repairs)
)

print(
    "Seeds with >= 2 legal depth-guard candidates:",
    int(
        pool_summary[
            "selection_comparison_possible"
        ].sum()
    )
)

print(
    "Seeds with only 1 candidate "
    "(random baseline == targeted):",
    int(
        (
            pool_summary[
                "num_legal_depth_guard_candidates"
            ] == 1
        ).sum()
    )
)

print(
    "\nTotal candidates in random-baseline pool:",
    len(random_baseline_pool)
)

,circuit,seed_plan_id,num_legal_depth_guard_candidates,num_non_target_candidates,targeted_candidate_id,selection_comparison_possible
0,mini_alu_305.qasm,P1,2,1,P1_N1,True
1,mini_alu_305.qasm,P11,2,1,P11_N1,True
2,mini_alu_305.qasm,P12,1,0,P12_N0,False
3,mini_alu_305.qasm,P13,1,0,P13_N0,False
4,mini_alu_305.qasm,P3,2,1,P3_N1,True
5,mini_alu_305.qasm,P4,1,0,P4_N0,False
6,mini_alu_305.qasm,P7,2,1,P7_N0,True
7,mini_alu_305.qasm,P9,1,0,P9_N0,False
8,rd73_140.qasm,P0,1,0,P0_N0,False
9,sym9_146.qasm,P5,1,0,P5_N0,False



Targeted interventions: 12
Seeds with >= 2 legal depth-guard candidates: 4
Seeds with only 1 candidate (random baseline == targeted): 8

Total candidates in random-baseline pool: 16


In [49]:
# ------------------------------------------------------------
# Seeds where targeted selection can actually be compared
# against another legal repair
# ------------------------------------------------------------

comparison_seeds = (
    pool_summary[
        pool_summary[
            "selection_comparison_possible"
        ]
    ][
        [
            "circuit",
            "seed_plan_id",
            "targeted_candidate_id",
        ]
    ]
    .copy()
)

alternative_repairs = (
    random_baseline_pool[
        (
            random_baseline_pool[
                "is_targeted_candidate"
            ] == False
        )
    ]
    .merge(
        comparison_seeds,
        on=[
            "circuit",
            "seed_plan_id",
            "targeted_candidate_id",
        ],
        how="inner",
    )
    .copy()
)

display(
    alternative_repairs[
        [
            "circuit",
            "seed_plan_id",
            "targeted_candidate_id",
            "candidate_plan_id",
            "max_weighted_degree_after",
            "weighted_degree_cv_after",
            "logical_depth_after",
        ]
    ]
)

print(
    "\nNumber of alternative repairs to route:",
    len(alternative_repairs)
)

assert len(alternative_repairs) == 4

,circuit,seed_plan_id,targeted_candidate_id,candidate_plan_id,max_weighted_degree_after,weighted_degree_cv_after,logical_depth_after
0,mini_alu_305.qasm,P1,P1_N1,P1_N0,43.0,0.445172,103
1,mini_alu_305.qasm,P11,P11_N1,P11_N0,44.0,0.438300,103
2,mini_alu_305.qasm,P3,P3_N1,P3_N0,43.0,0.431317,102
3,mini_alu_305.qasm,P7,P7_N0,P7_N1,44.0,0.438300,103



Number of alternative repairs to route: 4


In [50]:
# ------------------------------------------------------------
# Physical evaluation of NON-TARGET alternatives
# ------------------------------------------------------------

alternative_raw_rows = []
alternative_failures = []

for _, repair in alternative_repairs.iterrows():

    circuit_name = repair["circuit"]
    seed_plan_id = repair["seed_plan_id"]
    candidate_plan_id = repair["candidate_plan_id"]

    print(
        f"\nRouting alternative: "
        f"{circuit_name} | "
        f"{seed_plan_id} -> {candidate_plan_id}"
    )

    qc = circuit_cache[circuit_name]

    candidate_chains_raw = repair[
        "candidate_chains"
    ]

    if isinstance(
        candidate_chains_raw,
        str
    ):
        candidate_chains_raw = ast.literal_eval(
            candidate_chains_raw
        )

    candidate_chains = [
        tuple(chain)
        for chain in candidate_chains_raw
    ]

    try:

        candidate_dynamic, _ = (
            compile_fixed_active_plan(
                qc,
                candidate_chains
            )
        )

        candidate_norm = (
            normalize_active_reset(
                candidate_dynamic
            )
        )

    except Exception as e:

        alternative_failures.append({
            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "candidate_plan_id":
                candidate_plan_id,

            "stage":
                "compile",

            "error":
                repr(e),
        })

        continue

    # Sanity check
    assert (
        candidate_norm.depth()
        ==
        int(
            repair[
                "logical_depth_after"
            ]
        )
    )

    original_raw = (
        exp1_raw[
            (
                exp1_raw["circuit"]
                == circuit_name
            )
            &
            (
                exp1_raw["plan_id"]
                == seed_plan_id
            )
        ]
        .sort_values(
            "routing_seed"
        )
        .copy()
    )

    assert (
        original_raw[
            "routing_seed"
        ].tolist()
        ==
        ROUTING_SEEDS
    )

    original_by_seed = (
        original_raw
        .set_index(
            "routing_seed"
        )
    )

    for routing_seed in ROUTING_SEEDS:

        try:

            routed = transpile(
                candidate_norm,
                coupling_map=TARGET_COUPLING,
                routing_method="sabre",
                layout_method="sabre",
                optimization_level=1,
                seed_transpiler=routing_seed,
            )

        except Exception as e:

            alternative_failures.append({
                "circuit":
                    circuit_name,

                "seed_plan_id":
                    seed_plan_id,

                "candidate_plan_id":
                    candidate_plan_id,

                "routing_seed":
                    routing_seed,

                "stage":
                    "routing",

                "error":
                    repr(e),
            })

            continue

        before = original_by_seed.loc[
            routing_seed
        ]

        swaps_after = int(
            routed.count_ops().get(
                "swap",
                0
            )
        )

        depth_after = int(
            routed.depth()
        )

        alternative_raw_rows.append({

            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "candidate_plan_id":
                candidate_plan_id,

            "role":
                "alternative",

            "routing_seed":
                routing_seed,

            "swaps_before":
                int(
                    before[
                        "swap_count"
                    ]
                ),

            "swaps_after":
                swaps_after,

            "delta_swaps":
                swaps_after
                -
                int(
                    before[
                        "swap_count"
                    ]
                ),

            "routed_depth_before":
                int(
                    before[
                        "routed_depth"
                    ]
                ),

            "routed_depth_after":
                depth_after,

            "delta_routed_depth":
                depth_after
                -
                int(
                    before[
                        "routed_depth"
                    ]
                ),
        })

        print(
            f"  seed {routing_seed}: "
            f"SWAP {int(before['swap_count'])}"
            f"->{swaps_after}, "
            f"depth {int(before['routed_depth'])}"
            f"->{depth_after}"
        )


alternative_raw = pd.DataFrame(
    alternative_raw_rows
)

print(
    "\nCompleted alternative routing rows:",
    len(alternative_raw)
)

print(
    "Failures:",
    len(alternative_failures)
)

assert len(alternative_raw) == 20


Routing alternative: mini_alu_305.qasm | P1 -> P1_N0


2026-09-28 11:43:46,774 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:43:46,782 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:46,782 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:46,786 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:43:46,786 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:43:46,790 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 11:43:46,812 - INFO - Pass: SabreLayout - 18.88180 (ms)
2026-09-28 11:43:46,812 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:43:46,820 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:46,820 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:46,824 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:43:46,824 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:46,824 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:46,824 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:43:46,824 - INFO - Pas

  seed 42: SWAP 42->42, depth 158->159
  seed 43: SWAP 44->46, depth 158->160


2026-09-28 11:43:47,106 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:43:47,106 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:47,111 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:47,111 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:43:47,111 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:43:47,111 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 11:43:47,136 - INFO - Pass: SabreLayout - 25.21658 (ms)
2026-09-28 11:43:47,136 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:43:47,136 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:47,136 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:47,145 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:43:47,145 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:47,148 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:47,150 - INFO - Pass: Size - 2.52199 (ms)
2026-09-28 11:43:47,150 - INFO - Pas

  seed 44: SWAP 42->47, depth 157->160
  seed 45: SWAP 45->42, depth 160->159


2026-09-28 11:43:47,300 - INFO - Pass: CXCancellation - 2.01011 (ms)
2026-09-28 11:43:47,302 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:43:47,303 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:47,305 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:47,306 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:43:47,306 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:47,311 - INFO - Pass: Optimize1qGatesDecomposition - 2.65336 (ms)
2026-09-28 11:43:47,311 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 11:43:47,311 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:43:47,311 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:47,320 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:47,323 - INFO - Pass: Size - 3.17121 (ms)
2026-09-28 11:43:47,323 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:47,323 - INFO - Total Transpile Time - 136.44981 (ms)
2026-09-28 11:43:47,428 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
20

... (9 outputs left out)

2026-09-28 11:43:48,562 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:43:48,570 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:48,570 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:48,570 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:43:48,570 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:43:48,570 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 11:43:48,596 - INFO - Pass: SabreLayout - 17.09127 (ms)
2026-09-28 11:43:48,596 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 11:43:48,596 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:48,603 - INFO - Pass: HighLevelSynthesis - 7.58672 (ms)
2026-09-28 11:43:48,603 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 11:43:48,603 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:48,607 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:48,607 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 11:43:48,607 - INFO - Pas

  seed 42: SWAP 45->44, depth 157->157
  seed 43: SWAP 48->44, depth 157->157


2026-09-28 11:43:48,839 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:48,842 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 11:43:48,842 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 11:43:48,842 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 11:43:48,842 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 11:43:48,851 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:48,856 - INFO - Pass: Size - 2.04492 (ms)
2026-09-28 11:43:48,856 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 11:43:48,860 - INFO - Total Transpile Time - 114.66312 (ms)
2026-09-28 11:43:48,873 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 11:43:48,873 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 11:43:48,873 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 11:43:48,873 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 11:43:48,873 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 11:43:48,883 - INFO - Pass: Barrie

  seed 44: SWAP 45->46, depth 157->159
  seed 45: SWAP 45->44, depth 157->157
  seed 46: SWAP 45->44, depth 157->157

Completed alternative routing rows: 20
Failures: 0


In [51]:
# ------------------------------------------------------------
# Targeted vs alternative: same seed plan, same SABRE seed
# ------------------------------------------------------------

comparison_key_df = (
    comparison_seeds[
        [
            "circuit",
            "seed_plan_id",
        ]
    ]
)

targeted_raw_subset = (
    experiment3_raw_routing
    .merge(
        comparison_key_df,
        on=[
            "circuit",
            "seed_plan_id",
        ],
        how="inner",
    )
    .copy()
)

targeted_raw_subset[
    "role"
] = "targeted"


targeted_vs_alt = (
    targeted_raw_subset[
        [
            "circuit",
            "seed_plan_id",
            "candidate_plan_id",
            "routing_seed",
            "swaps_after",
            "routed_depth_after",
        ]
    ]
    .rename(
        columns={
            "candidate_plan_id":
                "targeted_candidate_id",

            "swaps_after":
                "targeted_swaps",

            "routed_depth_after":
                "targeted_depth",
        }
    )
    .merge(
        alternative_raw[
            [
                "circuit",
                "seed_plan_id",
                "candidate_plan_id",
                "routing_seed",
                "swaps_after",
                "routed_depth_after",
            ]
        ]
        .rename(
            columns={
                "candidate_plan_id":
                    "alternative_candidate_id",

                "swaps_after":
                    "alternative_swaps",

                "routed_depth_after":
                    "alternative_depth",
            }
        ),
        on=[
            "circuit",
            "seed_plan_id",
            "routing_seed",
        ],
        how="inner",
    )
)

targeted_vs_alt[
    "targeted_minus_alternative_swaps"
] = (
    targeted_vs_alt[
        "targeted_swaps"
    ]
    -
    targeted_vs_alt[
        "alternative_swaps"
    ]
)

targeted_vs_alt[
    "targeted_minus_alternative_depth"
] = (
    targeted_vs_alt[
        "targeted_depth"
    ]
    -
    targeted_vs_alt[
        "alternative_depth"
    ]
)

display(targeted_vs_alt)

,circuit,seed_plan_id,targeted_candidate_id,routing_seed,targeted_swaps,targeted_depth,alternative_candidate_id,alternative_swaps,alternative_depth,targeted_minus_alternative_swaps,targeted_minus_alternative_depth
0,mini_alu_305.qasm,P1,P1_N1,42,43,158,P1_N0,42,159,1,-1
1,mini_alu_305.qasm,P1,P1_N1,43,43,158,P1_N0,46,160,-3,-2
2,mini_alu_305.qasm,P1,P1_N1,44,43,159,P1_N0,47,160,-4,-1
3,mini_alu_305.qasm,P1,P1_N1,45,43,159,P1_N0,42,159,1,0
4,mini_alu_305.qasm,P1,P1_N1,46,43,159,P1_N0,43,159,0,0
5,mini_alu_305.qasm,P11,P11_N1,42,45,159,P11_N0,44,157,1,2
6,mini_alu_305.qasm,P11,P11_N1,43,47,162,P11_N0,44,157,3,5
7,mini_alu_305.qasm,P11,P11_N1,44,44,157,P11_N0,46,159,-2,-2
8,mini_alu_305.qasm,P11,P11_N1,45,46,157,P11_N0,44,157,2,0
9,mini_alu_305.qasm,P11,P11_N1,46,44,158,P11_N0,44,157,0,1


In [52]:
selection_summary = (
    targeted_vs_alt
    .groupby(
        [
            "circuit",
            "seed_plan_id",
            "targeted_candidate_id",
            "alternative_candidate_id",
        ]
    )
    .agg(
        targeted_mean_swaps=(
            "targeted_swaps",
            "mean"
        ),

        alternative_mean_swaps=(
            "alternative_swaps",
            "mean"
        ),

        targeted_mean_depth=(
            "targeted_depth",
            "mean"
        ),

        alternative_mean_depth=(
            "alternative_depth",
            "mean"
        ),
    )
    .reset_index()
)

selection_summary[
    "targeted_swap_advantage"
] = (
    selection_summary[
        "alternative_mean_swaps"
    ]
    -
    selection_summary[
        "targeted_mean_swaps"
    ]
)

selection_summary[
    "targeted_depth_advantage"
] = (
    selection_summary[
        "alternative_mean_depth"
    ]
    -
    selection_summary[
        "targeted_mean_depth"
    ]
)

display(selection_summary)

,circuit,seed_plan_id,targeted_candidate_id,alternative_candidate_id,targeted_mean_swaps,alternative_mean_swaps,targeted_mean_depth,alternative_mean_depth,targeted_swap_advantage,targeted_depth_advantage
0,mini_alu_305.qasm,P1,P1_N1,P1_N0,43.0,44.0,158.6,159.4,1.0,0.8
1,mini_alu_305.qasm,P11,P11_N1,P11_N0,45.2,44.4,158.6,157.4,-0.8,-1.2
2,mini_alu_305.qasm,P3,P3_N1,P3_N0,45.4,45.2,157.2,159.8,-0.2,2.6
3,mini_alu_305.qasm,P7,P7_N0,P7_N1,45.2,44.4,158.6,157.4,-0.8,-1.2


In [53]:
# ------------------------------------------------------------
# Exact expected performance of UNIFORM RANDOM repair
# ------------------------------------------------------------

random_exact = (
    targeted_vs_alt.copy()
)

random_exact[
    "random_expected_swaps"
] = (
    random_exact[
        "targeted_swaps"
    ]
    +
    random_exact[
        "alternative_swaps"
    ]
) / 2.0

random_exact[
    "random_expected_depth"
] = (
    random_exact[
        "targeted_depth"
    ]
    +
    random_exact[
        "alternative_depth"
    ]
) / 2.0


random_exact[
    "targeted_vs_random_swaps"
] = (
    random_exact[
        "targeted_swaps"
    ]
    -
    random_exact[
        "random_expected_swaps"
    ]
)

random_exact[
    "targeted_vs_random_depth"
] = (
    random_exact[
        "targeted_depth"
    ]
    -
    random_exact[
        "random_expected_depth"
    ]
)


random_summary = (
    random_exact
    .groupby(
        [
            "circuit",
            "seed_plan_id",
        ]
    )
    .agg(
        targeted_mean_swaps=(
            "targeted_swaps",
            "mean"
        ),

        random_expected_mean_swaps=(
            "random_expected_swaps",
            "mean"
        ),

        targeted_mean_depth=(
            "targeted_depth",
            "mean"
        ),

        random_expected_mean_depth=(
            "random_expected_depth",
            "mean"
        ),
    )
    .reset_index()
)


random_summary[
    "targeted_vs_random_swap_advantage"
] = (
    random_summary[
        "random_expected_mean_swaps"
    ]
    -
    random_summary[
        "targeted_mean_swaps"
    ]
)

random_summary[
    "targeted_vs_random_depth_advantage"
] = (
    random_summary[
        "random_expected_mean_depth"
    ]
    -
    random_summary[
        "targeted_mean_depth"
    ]
)

display(random_summary)

,circuit,seed_plan_id,targeted_mean_swaps,random_expected_mean_swaps,targeted_mean_depth,random_expected_mean_depth,targeted_vs_random_swap_advantage,targeted_vs_random_depth_advantage
0,mini_alu_305.qasm,P1,43.0,43.5,158.6,159.0,0.5,0.4
1,mini_alu_305.qasm,P11,45.2,44.8,158.6,158.0,-0.4,-0.6
2,mini_alu_305.qasm,P3,45.4,45.3,157.2,158.5,-0.1,1.3
3,mini_alu_305.qasm,P7,45.2,44.8,158.6,158.0,-0.4,-0.6


In [54]:
print(
    "Selection-comparable seeds:",
    len(random_summary)
)

print(
    "\nTargeted beats alternative in mean SWAP:"
)

print(
    int(
        (
            selection_summary[
                "targeted_swap_advantage"
            ] > 0
        ).sum()
    ),
    "/",
    len(selection_summary)
)

print(
    "Targeted beats alternative in mean routed depth:"
)

print(
    int(
        (
            selection_summary[
                "targeted_depth_advantage"
            ] > 0
        ).sum()
    ),
    "/",
    len(selection_summary)
)


print(
    "\nMean targeted-vs-random SWAP advantage:"
)

print(
    random_summary[
        "targeted_vs_random_swap_advantage"
    ].mean()
)

print(
    "Mean targeted-vs-random depth advantage:"
)

print(
    random_summary[
        "targeted_vs_random_depth_advantage"
    ].mean()
)

Selection-comparable seeds: 4

Targeted beats alternative in mean SWAP:
1 / 4
Targeted beats alternative in mean routed depth:
2 / 4

Mean targeted-vs-random SWAP advantage:
-0.1000000000000032
Mean targeted-vs-random depth advantage:
0.1250000000000071


## Experiment 3A.2 — Generalized Local Repair

The strict two-chain suffix-swap operator required both chains to have a
non-empty suffix. This made many local neighborhoods very small.

We now generalize the operator by allowing one suffix to be empty.

Example:

A = (9 | )
B = (2 | 4, 6)

becomes:

A' = (9, 4, 6)
B' = (2)

The number of reuse chains remains unchanged, so reuse width is preserved.

Every generated candidate must still satisfy the full GidNET legality and
dynamic-circuit validation used previously.

No physical routing is used during candidate generation or structural
selection.

In [57]:
# ------------------------------------------------------------
# Generalized two-chain suffix exchange
#
# Allows ONE suffix to be empty.
#
# Example:
#
# A = (9 | )
# B = (2 | 4, 6)
#
# ->
#
# A' = (9, 4, 6)
# B' = (2,)
#
# Both prefixes must remain non-empty.
# Width is therefore preserved.
# ------------------------------------------------------------

def generate_generalized_hotspot_neighbors(
    circuit,
    seed_chains,
    hotspot_indices,
):

    seed_chains = [
        tuple(chain)
        for chain in seed_chains
    ]

    seed_signature = (
        canonical_plan_signature(
            seed_chains
        )
    )

    seen = {
        seed_signature
    }

    legal_neighbors = []

    rejection_reasons = defaultdict(int)

    generated_raw = 0

    n = len(seed_chains)

    # --------------------------------------------------------
    # At least one hotspot chain must participate
    # --------------------------------------------------------

    for hotspot_idx in hotspot_indices:

        A = seed_chains[
            hotspot_idx
        ]

        for other_idx in range(n):

            if other_idx == hotspot_idx:
                continue

            B = seed_chains[
                other_idx
            ]

            # ------------------------------------------------
            # Generalized cuts
            #
            # OLD:
            #   range(1, len(chain))
            #
            # NEW:
            #   range(1, len(chain) + 1)
            #
            # cut == len(chain)
            # means EMPTY suffix.
            #
            # Prefix is always non-empty.
            # ------------------------------------------------

            for cut_A in range(
                1,
                len(A) + 1
            ):

                for cut_B in range(
                    1,
                    len(B) + 1
                ):

                    prefix_A = A[:cut_A]
                    suffix_A = A[cut_A:]

                    prefix_B = B[:cut_B]
                    suffix_B = B[cut_B:]

                    # ----------------------------------------
                    # Both suffixes empty => no modification
                    # ----------------------------------------

                    if (
                        len(suffix_A) == 0
                        and
                        len(suffix_B) == 0
                    ):
                        rejection_reasons[
                            "no_change"
                        ] += 1

                        continue

                    generated_raw += 1

                    # ----------------------------------------
                    # Exchange suffixes
                    # ----------------------------------------

                    new_A = (
                        prefix_A
                        +
                        suffix_B
                    )

                    new_B = (
                        prefix_B
                        +
                        suffix_A
                    )

                    # Prefixes were required to be non-empty,
                    # therefore neither resulting chain can
                    # become empty.
                    assert len(new_A) > 0
                    assert len(new_B) > 0

                    candidate = [
                        tuple(chain)
                        for chain in seed_chains
                    ]

                    candidate[
                        hotspot_idx
                    ] = new_A

                    candidate[
                        other_idx
                    ] = new_B

                    # ----------------------------------------
                    # Width must remain constant
                    # ----------------------------------------

                    if (
                        len(candidate)
                        != len(seed_chains)
                    ):
                        rejection_reasons[
                            "width_changed"
                        ] += 1

                        continue

                    # ----------------------------------------
                    # Deduplicate
                    # ----------------------------------------

                    signature = (
                        canonical_plan_signature(
                            candidate
                        )
                    )

                    if signature in seen:

                        rejection_reasons[
                            "duplicate"
                        ] += 1

                        continue

                    seen.add(
                        signature
                    )

                    # ----------------------------------------
                    # Full GidNET legality validation
                    # ----------------------------------------

                    validation = (
                        validate_active_plan(
                            circuit,
                            candidate
                        )
                    )

                    if not validation[
                        "legal"
                    ]:

                        rejection_reasons[
                            validation["reason"]
                        ] += 1

                        continue

                    # ----------------------------------------
                    # Identify move type
                    # ----------------------------------------

                    if (
                        len(suffix_A) == 0
                        or
                        len(suffix_B) == 0
                    ):

                        move_type = (
                            "one_sided_suffix_transfer"
                        )

                    else:

                        move_type = (
                            "two_sided_suffix_swap"
                        )

                    # ----------------------------------------
                    # Boundary difference
                    # ----------------------------------------

                    before_boundaries = set(
                        boundaries_from_chains(
                            seed_chains
                        )
                    )

                    after_boundaries = set(
                        boundaries_from_chains(
                            candidate
                        )
                    )

                    removed_boundaries = sorted(
                        before_boundaries
                        -
                        after_boundaries
                    )

                    added_boundaries = sorted(
                        after_boundaries
                        -
                        before_boundaries
                    )

                    legal_neighbors.append({

                        "hotspot_chain_idx":
                            hotspot_idx,

                        "other_chain_idx":
                            other_idx,

                        "cut_hotspot":
                            cut_A,

                        "cut_other":
                            cut_B,

                        "move_type":
                            move_type,

                        "candidate_chains":
                            candidate,

                        "plan_signature":
                            signature,

                        "removed_boundaries":
                            removed_boundaries,

                        "added_boundaries":
                            added_boundaries,

                        "validation":
                            validation,
                    })

    return {

        "generated_raw":
            generated_raw,

        "legal_neighbors":
            legal_neighbors,

        "rejection_reasons":
            dict(
                rejection_reasons
            ),
    }

In [58]:
# ------------------------------------------------------------
# Sanity test:
# system9 / P0 was impossible under strict suffix-swap
# because its hotspot was singleton (9,)
# ------------------------------------------------------------

p0_generalized = (
    generate_generalized_hotspot_neighbors(
        original_circuit,
        example_chains,
        p0_metrics[
            "hotspot_indices"
        ],
    )
)

print(
    "Seed hotspot:",
    [
        example_chains[i]
        for i in p0_metrics[
            "hotspot_indices"
        ]
    ]
)

print(
    "\nGeneralized raw attempts:",
    p0_generalized[
        "generated_raw"
    ]
)

print(
    "Generalized legal neighbors:",
    len(
        p0_generalized[
            "legal_neighbors"
        ]
    )
)

print(
    "\nRejection reasons:"
)

print(
    p0_generalized[
        "rejection_reasons"
    ]
)


for i, neighbor in enumerate(
    p0_generalized[
        "legal_neighbors"
    ]
):

    print(
        "\n"
        + "=" * 65
    )

    print(
        f"Neighbor G{i}"
    )

    print(
        "Move type:",
        neighbor[
            "move_type"
        ]
    )

    print(
        "Partner chain:",
        neighbor[
            "other_chain_idx"
        ]
    )

    print(
        "Cuts:",
        neighbor[
            "cut_hotspot"
        ],
        neighbor[
            "cut_other"
        ]
    )

    print(
        "Removed boundaries:",
        neighbor[
            "removed_boundaries"
        ]
    )

    print(
        "Added boundaries:",
        neighbor[
            "added_boundaries"
        ]
    )

    print(
        "Candidate chains:"
    )

    for chain in neighbor[
        "candidate_chains"
    ]:

        print(
            " ",
            chain
        )

Seed hotspot: [(9,)]

Generalized raw attempts: 5
Generalized legal neighbors: 0

Rejection reasons:
{'boundary_not_in_candidate_matrix': 5, 'no_change': 6}


In [61]:
# ------------------------------------------------------------
# Generalized neighborhood coverage across all 50 seed plans
# NO routing
# ------------------------------------------------------------

generalized_coverage_rows = []

for _, row in exp2_plans.iterrows():

    circuit_name = (
        row["circuit"]
    )

    plan_id = (
        row["plan_id"]
    )

    qc = circuit_cache[
        circuit_name
    ]

    chains = [
        tuple(chain)
        for chain in parse_saved_structure(
            row["reuse_chains"]
        )
    ]

    metrics = (
        compute_quotient_load_metrics(
            qc,
            chains
        )
    )

    neighborhood = (
        generate_generalized_hotspot_neighbors(
            qc,
            chains,
            metrics[
                "hotspot_indices"
            ],
        )
    )

    legal_neighbors = (
        neighborhood[
            "legal_neighbors"
        ]
    )

    num_one_sided = sum(
        neighbor[
            "move_type"
        ]
        ==
        "one_sided_suffix_transfer"

        for neighbor in legal_neighbors
    )

    num_two_sided = sum(
        neighbor[
            "move_type"
        ]
        ==
        "two_sided_suffix_swap"

        for neighbor in legal_neighbors
    )

    generalized_coverage_rows.append({

        "circuit":
            circuit_name,

        "plan_id":
            plan_id,

        "width":
            int(row["width"]),

        "hotspot_chains":
            tuple(
                chains[i]
                for i in metrics[
                    "hotspot_indices"
                ]
            ),

        "generalized_raw_attempts":
            neighborhood[
                "generated_raw"
            ],

        "generalized_legal_neighbors":
            len(
                legal_neighbors
            ),

        "legal_one_sided":
            num_one_sided,

        "legal_two_sided":
            num_two_sided,

        "has_generalized_neighbor":
            len(
                legal_neighbors
            ) > 0,
    })


generalized_coverage = pd.DataFrame(
    generalized_coverage_rows
)

print(
    "Total seed plans:",
    len(
        generalized_coverage
    )
)

print(
    "\nSeeds with >=1 generalized legal neighbor:",
    int(
        generalized_coverage[
            "has_generalized_neighbor"
        ].sum()
    )
)

print(
    "\nTotal generalized legal neighbors:",
    int(
        generalized_coverage[
            "generalized_legal_neighbors"
        ].sum()
    )
)

print(
    "Total legal one-sided transfers:",
    int(
        generalized_coverage[
            "legal_one_sided"
        ].sum()
    )
)

print(
    "Total legal two-sided swaps:",
    int(
        generalized_coverage[
            "legal_two_sided"
        ].sum()
    )
)

Total seed plans: 50

Seeds with >=1 generalized legal neighbor: 40

Total generalized legal neighbors: 65
Total legal one-sided transfers: 44
Total legal two-sided swaps: 21


In [62]:
# ------------------------------------------------------------
# Strict vs generalized neighborhood coverage
# ------------------------------------------------------------

comparison = (
    experiment3_screening[
        [
            "circuit",
            "seed_plan_id",
            "dag_legal_neighbors",
        ]
    ]
    .rename(
        columns={
            "seed_plan_id":
                "plan_id",

            "dag_legal_neighbors":
                "strict_legal_neighbors",
        }
    )
    .merge(
        generalized_coverage[
            [
                "circuit",
                "plan_id",
                "generalized_legal_neighbors",
                "legal_one_sided",
                "legal_two_sided",
            ]
        ],
        on=[
            "circuit",
            "plan_id",
        ],
        how="inner",
    )
)

comparison[
    "neighbor_gain"
] = (
    comparison[
        "generalized_legal_neighbors"
    ]
    -
    comparison[
        "strict_legal_neighbors"
    ]
)

comparison[
    "strict_has_neighbor"
] = (
    comparison[
        "strict_legal_neighbors"
    ] > 0
)

comparison[
    "generalized_has_neighbor"
] = (
    comparison[
        "generalized_legal_neighbors"
    ] > 0
)


print(
    "Strict operator coverage:",
    int(
        comparison[
            "strict_has_neighbor"
        ].sum()
    ),
    "/",
    len(comparison)
)

print(
    "Generalized operator coverage:",
    int(
        comparison[
            "generalized_has_neighbor"
        ].sum()
    ),
    "/",
    len(comparison)
)

print(
    "\nPreviously unreachable seeds "
    "that now have a legal repair:"
)

newly_reachable = (
    comparison[
        (~comparison[
            "strict_has_neighbor"
        ])
        &
        (
            comparison[
                "generalized_has_neighbor"
            ]
        )
    ]
)

print(
    len(
        newly_reachable
    )
)

display(
    newly_reachable[
        [
            "circuit",
            "plan_id",
            "strict_legal_neighbors",
            "generalized_legal_neighbors",
            "legal_one_sided",
        ]
    ]
)

Strict operator coverage: 17 / 50
Generalized operator coverage: 40 / 50

Previously unreachable seeds that now have a legal repair:
23


,circuit,plan_id,strict_legal_neighbors,generalized_legal_neighbors,legal_one_sided
4,system9.qasm,P12,0,3,3
5,system9.qasm,P2,0,3,3
6,system9.qasm,P3,0,1,1
10,system9.qasm,P7,0,1,1
12,system9.qasm,P9,0,1,1
14,sym9_146.qasm,P1,0,1,1
15,sym9_146.qasm,P2,0,1,1
16,sym9_146.qasm,P3,0,1,1
17,sym9_146.qasm,P4,0,1,1
19,sym9_146.qasm,P6,0,1,1


In [63]:
generalized_summary_by_circuit = (
    generalized_coverage
    .groupby(
        "circuit"
    )
    .agg(

        total_seeds=(
            "plan_id",
            "size"
        ),

        seeds_with_legal_neighbor=(
            "has_generalized_neighbor",
            "sum"
        ),

        total_legal_neighbors=(
            "generalized_legal_neighbors",
            "sum"
        ),

        total_one_sided=(
            "legal_one_sided",
            "sum"
        ),

        total_two_sided=(
            "legal_two_sided",
            "sum"
        ),
    )
)

display(
    generalized_summary_by_circuit
)

,total_seeds,seeds_with_legal_neighbor,total_legal_neighbors,total_one_sided,total_two_sided
circuit,,,,,
alu-bdd_288.qasm,6,6,8,8,0
mini_alu_305.qasm,16,16,27,12,15
rd73_140.qasm,6,2,2,0,2
sym9_146.qasm,9,9,11,9,2
system9.qasm,13,7,17,15,2


In [64]:
# ------------------------------------------------------------
# Experiment 3A.2
# Full structural screening of GENERALIZED repair neighborhood
# NO PHYSICAL ROUTING
# ------------------------------------------------------------

generalized_candidate_rows = []
generalized_seed_rows = []

for _, row in exp2_plans.iterrows():

    circuit_name = row["circuit"]
    plan_id = row["plan_id"]

    qc = circuit_cache[circuit_name]

    seed_chains = [
        tuple(chain)
        for chain in parse_saved_structure(
            row["reuse_chains"]
        )
    ]

    seed_width = int(
        row["width"]
    )

    seed_depth = int(
        row["normalized_logical_depth"]
    )

    # --------------------------------------------------------
    # Seed hotspot metrics
    # --------------------------------------------------------

    seed_metrics = (
        compute_quotient_load_metrics(
            qc,
            seed_chains
        )
    )

    seed_max = (
        seed_metrics[
            "max_weighted_degree"
        ]
    )

    seed_cv = (
        seed_metrics[
            "weighted_degree_cv"
        ]
    )

    hotspot_indices = (
        seed_metrics[
            "hotspot_indices"
        ]
    )

    # --------------------------------------------------------
    # Generate generalized legal neighborhood
    # --------------------------------------------------------

    neighborhood = (
        generate_generalized_hotspot_neighbors(
            qc,
            seed_chains,
            hotspot_indices
        )
    )

    legal_neighbors = (
        neighborhood[
            "legal_neighbors"
        ]
    )

    local_rows = []

    compile_failures = 0
    width_failures = 0

    # --------------------------------------------------------
    # Full compilation + cheap structural scoring
    # --------------------------------------------------------

    for local_idx, neighbor in enumerate(
        legal_neighbors
    ):

        candidate_id = (
            f"{plan_id}_G{local_idx}"
        )

        candidate_chains = (
            neighbor[
                "candidate_chains"
            ]
        )

        try:

            candidate_dynamic, _ = (
                compile_fixed_active_plan(
                    qc,
                    candidate_chains
                )
            )

            candidate_width = (
                get_active_qubit_count(
                    candidate_dynamic
                )
            )

            candidate_norm = (
                normalize_active_reset(
                    candidate_dynamic
                )
            )

            candidate_depth = (
                candidate_norm.depth()
            )

        except Exception:

            compile_failures += 1
            continue

        # ----------------------------------------------------
        # Width preservation
        # ----------------------------------------------------

        if candidate_width != seed_width:

            width_failures += 1
            continue

        # ----------------------------------------------------
        # Experiment-2 hotspot metrics
        # ----------------------------------------------------

        metrics = (
            compute_quotient_load_metrics(
                qc,
                candidate_chains
            )
        )

        candidate_max = (
            metrics[
                "max_weighted_degree"
            ]
        )

        candidate_cv = (
            metrics[
                "weighted_degree_cv"
            ]
        )

        delta_max = (
            candidate_max
            - seed_max
        )

        delta_cv = (
            candidate_cv
            - seed_cv
        )

        # ----------------------------------------------------
        # Logical-depth guard
        # ----------------------------------------------------

        relative_depth_change = (
            candidate_depth
            - seed_depth
        ) / seed_depth

        depth_guard_pass = (
            relative_depth_change
            <= LOGICAL_DEPTH_TOLERANCE
        )

        # ----------------------------------------------------
        # Structural hotspot improvement
        # ----------------------------------------------------

        hotspot_improved = (
            candidate_max < seed_max
        ) or (
            np.isclose(
                candidate_max,
                seed_max
            )
            and
            candidate_cv < seed_cv
        )

        record = {

            "circuit":
                circuit_name,

            "seed_plan_id":
                plan_id,

            "candidate_plan_id":
                candidate_id,

            "move_type":
                neighbor[
                    "move_type"
                ],

            "candidate_chains":
                candidate_chains,

            "hotspot_chain_idx":
                neighbor[
                    "hotspot_chain_idx"
                ],

            "other_chain_idx":
                neighbor[
                    "other_chain_idx"
                ],

            "cut_hotspot":
                neighbor[
                    "cut_hotspot"
                ],

            "cut_other":
                neighbor[
                    "cut_other"
                ],

            "removed_boundaries":
                neighbor[
                    "removed_boundaries"
                ],

            "added_boundaries":
                neighbor[
                    "added_boundaries"
                ],

            "width_before":
                seed_width,

            "width_after":
                candidate_width,

            "logical_depth_before":
                seed_depth,

            "logical_depth_after":
                candidate_depth,

            "relative_depth_change":
                relative_depth_change,

            "depth_guard_pass":
                depth_guard_pass,

            "max_weighted_degree_before":
                seed_max,

            "max_weighted_degree_after":
                candidate_max,

            "delta_max_weighted_degree":
                delta_max,

            "weighted_degree_cv_before":
                seed_cv,

            "weighted_degree_cv_after":
                candidate_cv,

            "delta_weighted_degree_cv":
                delta_cv,

            "hotspot_improved":
                hotspot_improved,

            "selected":
                False,
        }

        local_rows.append(
            record
        )

        generalized_candidate_rows.append(
            record
        )

    # --------------------------------------------------------
    # Choose targeted generalized repair
    # --------------------------------------------------------

    local_df = pd.DataFrame(
        local_rows
    )

    eligible_count = 0
    selected_id = None
    selected_move_type = None

    if len(local_df) > 0:

        eligible_df = (
            local_df[
                local_df[
                    "hotspot_improved"
                ]
                &
                local_df[
                    "depth_guard_pass"
                ]
            ]
            .copy()
        )

        eligible_count = len(
            eligible_df
        )

        if eligible_count > 0:

            # Same lexicographic criterion
            eligible_df = (
                eligible_df
                .sort_values(
                    [
                        "max_weighted_degree_after",
                        "weighted_degree_cv_after",
                    ]
                )
            )

            best = (
                eligible_df.iloc[0]
            )

            selected_id = (
                best[
                    "candidate_plan_id"
                ]
            )

            selected_move_type = (
                best[
                    "move_type"
                ]
            )

            # Mark selected master record
            for rec in generalized_candidate_rows[::-1]:

                if (
                    rec["circuit"]
                    == circuit_name
                    and
                    rec["seed_plan_id"]
                    == plan_id
                    and
                    rec["candidate_plan_id"]
                    == selected_id
                ):

                    rec["selected"] = True
                    break

    # --------------------------------------------------------
    # Seed summary
    # --------------------------------------------------------

    generalized_seed_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            plan_id,

        "width":
            seed_width,

        "raw_attempts":
            neighborhood[
                "generated_raw"
            ],

        "dag_legal_neighbors":
            len(
                legal_neighbors
            ),

        "fully_compiled_neighbors":
            len(
                local_rows
            ),

        "compile_failures":
            compile_failures,

        "width_failures":
            width_failures,

        "eligible_neighbors":
            eligible_count,

        "eligible_seed":
            selected_id
            is not None,

        "selected_candidate_id":
            selected_id,

        "selected_move_type":
            selected_move_type,
    })


generalized_candidates = pd.DataFrame(
    generalized_candidate_rows
)

generalized_screening = pd.DataFrame(
    generalized_seed_rows
)

print(
    "Generalized structural screening complete."
)

Generalized structural screening complete.


In [65]:
print(
    "Total seeds:",
    len(generalized_screening)
)

print(
    "\nSeeds with >=1 legal generalized repair:",
    int(
        (
            generalized_screening[
                "dag_legal_neighbors"
            ] > 0
        ).sum()
    )
)

print(
    "Seeds with >=1 fully compiled repair:",
    int(
        (
            generalized_screening[
                "fully_compiled_neighbors"
            ] > 0
        ).sum()
    )
)

print(
    "\nELIGIBLE generalized seeds:",
    int(
        generalized_screening[
            "eligible_seed"
        ].sum()
    )
)

print(
    "\nPrevious strict eligible seeds:",
    int(
        experiment3_screening[
            "eligible_seed"
        ].sum()
    )
)

Total seeds: 50

Seeds with >=1 legal generalized repair: 40
Seeds with >=1 fully compiled repair: 40

ELIGIBLE generalized seeds: 24

Previous strict eligible seeds: 12


In [66]:
generalized_eligibility_by_circuit = (
    generalized_screening
    .groupby("circuit")
    .agg(
        total_seeds=(
            "seed_plan_id",
            "size"
        ),

        has_legal_repair=(
            "dag_legal_neighbors",
            lambda x:
                int((x > 0).sum())
        ),

        eligible_seeds=(
            "eligible_seed",
            "sum"
        ),
    )
)

display(
    generalized_eligibility_by_circuit
)

,total_seeds,has_legal_repair,eligible_seeds
circuit,,,
alu-bdd_288.qasm,6,6,2
mini_alu_305.qasm,16,16,12
rd73_140.qasm,6,2,1
sym9_146.qasm,9,9,5
system9.qasm,13,7,4


In [67]:
generalized_selected = (
    generalized_candidates[
        generalized_candidates[
            "selected"
        ] == True
    ]
    .copy()
)

print(
    "Selected generalized repairs:",
    len(
        generalized_selected
    )
)

print(
    "\nSelected repair types:"
)

print(
    generalized_selected[
        "move_type"
    ].value_counts()
)

display(
    generalized_selected[
        [
            "circuit",
            "seed_plan_id",
            "candidate_plan_id",
            "move_type",

            "max_weighted_degree_before",
            "max_weighted_degree_after",
            "delta_max_weighted_degree",

            "weighted_degree_cv_before",
            "weighted_degree_cv_after",

            "logical_depth_before",
            "logical_depth_after",
            "relative_depth_change",
        ]
    ]
    .sort_values(
        [
            "circuit",
            "seed_plan_id",
        ]
    )
)

Selected generalized repairs: 24

Selected repair types:
move_type
one_sided_suffix_transfer    20
two_sided_suffix_swap         4
Name: count, dtype: int64


,circuit,seed_plan_id,candidate_plan_id,move_type,max_weighted_degree_before,max_weighted_degree_after,delta_max_weighted_degree,weighted_degree_cv_before,weighted_degree_cv_after,logical_depth_before,logical_depth_after,relative_depth_change
58,alu-bdd_288.qasm,P0,P0_G1,one_sided_suffix_transfer,25.0,20.0,-5.0,0.429198,0.251037,51,50,-0.019608
63,alu-bdd_288.qasm,P5,P5_G0,one_sided_suffix_transfer,25.0,20.0,-5.0,0.339055,0.192483,51,52,0.019608
32,mini_alu_305.qasm,P1,P1_G1,two_sided_suffix_swap,44.0,40.0,-4.0,0.450820,0.425411,103,103,0.000000
35,mini_alu_305.qasm,P11,P11_G1,two_sided_suffix_swap,43.0,39.0,-4.0,0.431317,0.407180,102,102,0.000000
37,mini_alu_305.qasm,P12,P12_G1,one_sided_suffix_transfer,44.0,33.0,-11.0,0.372797,0.182281,103,103,0.000000
39,mini_alu_305.qasm,P13,P13_G1,one_sided_suffix_transfer,54.0,41.0,-13.0,0.540172,0.400287,98,101,0.030612
40,mini_alu_305.qasm,P14,P14_G0,one_sided_suffix_transfer,42.0,41.0,-1.0,0.340151,0.324935,98,101,0.030612
43,mini_alu_305.qasm,P2,P2_G1,one_sided_suffix_transfer,39.0,31.0,-8.0,0.309789,0.141672,94,94,0.000000
45,mini_alu_305.qasm,P3,P3_G1,two_sided_suffix_swap,44.0,40.0,-4.0,0.438300,0.412120,103,103,0.000000
47,mini_alu_305.qasm,P4,P4_G1,one_sided_suffix_transfer,40.0,33.0,-7.0,0.342375,0.183664,95,95,0.000000


In [68]:
strict_vs_generalized = (
    experiment3_screening[
        [
            "circuit",
            "seed_plan_id",
            "eligible_seed",
        ]
    ]
    .rename(
        columns={
            "eligible_seed":
                "strict_eligible"
        }
    )
    .merge(
        generalized_screening[
            [
                "circuit",
                "seed_plan_id",
                "eligible_seed",
                "selected_move_type",
            ]
        ]
        .rename(
            columns={
                "eligible_seed":
                    "generalized_eligible"
            }
        ),
        on=[
            "circuit",
            "seed_plan_id",
        ],
        how="inner",
    )
)

newly_eligible = (
    strict_vs_generalized[
        (~strict_vs_generalized[
            "strict_eligible"
        ])
        &
        (
            strict_vs_generalized[
                "generalized_eligible"
            ]
        )
    ]
)

print(
    "Strict eligible seeds:",
    int(
        strict_vs_generalized[
            "strict_eligible"
        ].sum()
    )
)

print(
    "Generalized eligible seeds:",
    int(
        strict_vs_generalized[
            "generalized_eligible"
        ].sum()
    )
)

print(
    "Newly eligible because of generalized repair:",
    len(
        newly_eligible
    )
)

display(
    newly_eligible
)

Strict eligible seeds: 12
Generalized eligible seeds: 24
Newly eligible because of generalized repair: 12


,circuit,seed_plan_id,strict_eligible,generalized_eligible,selected_move_type
4,system9.qasm,P12,False,True,one_sided_suffix_transfer
5,system9.qasm,P2,False,True,one_sided_suffix_transfer
13,sym9_146.qasm,P0,False,True,one_sided_suffix_transfer
16,sym9_146.qasm,P3,False,True,one_sided_suffix_transfer
19,sym9_146.qasm,P6,False,True,one_sided_suffix_transfer
20,sym9_146.qasm,P7,False,True,one_sided_suffix_transfer
34,mini_alu_305.qasm,P14,False,True,one_sided_suffix_transfer
36,mini_alu_305.qasm,P2,False,True,one_sided_suffix_transfer
40,mini_alu_305.qasm,P6,False,True,one_sided_suffix_transfer
42,mini_alu_305.qasm,P8,False,True,one_sided_suffix_transfer


In [69]:
# ------------------------------------------------------------
# Physical evaluation of all selected GENERALIZED repairs
# ------------------------------------------------------------

generalized_raw_rows = []
generalized_routing_failures = []

for _, repair in generalized_selected.iterrows():

    circuit_name = repair["circuit"]
    seed_plan_id = repair["seed_plan_id"]
    candidate_plan_id = repair["candidate_plan_id"]

    print(
        f"\nRouting generalized repair: "
        f"{circuit_name} | "
        f"{seed_plan_id} -> {candidate_plan_id}"
    )

    qc = circuit_cache[circuit_name]

    candidate_chains_raw = repair["candidate_chains"]

    if isinstance(candidate_chains_raw, str):
        candidate_chains_raw = ast.literal_eval(
            candidate_chains_raw
        )

    candidate_chains = [
        tuple(chain)
        for chain in candidate_chains_raw
    ]

    # ----------------------------------------
    # Compile repaired plan
    # ----------------------------------------

    try:

        candidate_dynamic, _ = (
            compile_fixed_active_plan(
                qc,
                candidate_chains
            )
        )

        candidate_norm = (
            normalize_active_reset(
                candidate_dynamic
            )
        )

    except Exception as e:

        generalized_routing_failures.append({
            "circuit": circuit_name,
            "seed_plan_id": seed_plan_id,
            "candidate_plan_id": candidate_plan_id,
            "stage": "compile",
            "error": repr(e),
        })

        continue

    # Same depth as structural screening
    assert (
        candidate_norm.depth()
        ==
        int(repair["logical_depth_after"])
    )

    # ----------------------------------------
    # Original seed plan results
    # already available from Experiment 1
    # ----------------------------------------

    original_raw = (
        exp1_raw[
            (exp1_raw["circuit"] == circuit_name)
            &
            (exp1_raw["plan_id"] == seed_plan_id)
        ]
        .sort_values("routing_seed")
        .copy()
    )

    assert (
        original_raw["routing_seed"].tolist()
        ==
        ROUTING_SEEDS
    )

    original_by_seed = (
        original_raw
        .set_index("routing_seed")
    )

    # ----------------------------------------
    # Route candidate with paired SABRE seeds
    # ----------------------------------------

    for routing_seed in ROUTING_SEEDS:

        try:

            routed = transpile(
                candidate_norm,
                coupling_map=TARGET_COUPLING,
                routing_method="sabre",
                layout_method="sabre",
                optimization_level=1,
                seed_transpiler=routing_seed,
            )

        except Exception as e:

            generalized_routing_failures.append({
                "circuit": circuit_name,
                "seed_plan_id": seed_plan_id,
                "candidate_plan_id": candidate_plan_id,
                "routing_seed": routing_seed,
                "stage": "routing",
                "error": repr(e),
            })

            continue

        before = original_by_seed.loc[
            routing_seed
        ]

        swaps_before = int(
            before["swap_count"]
        )

        depth_before = int(
            before["routed_depth"]
        )

        swaps_after = int(
            routed.count_ops().get(
                "swap",
                0
            )
        )

        depth_after = int(
            routed.depth()
        )

        generalized_raw_rows.append({

            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "candidate_plan_id":
                candidate_plan_id,

            "move_type":
                repair["move_type"],

            "routing_seed":
                routing_seed,

            "logical_depth_before":
                int(
                    repair[
                        "logical_depth_before"
                    ]
                ),

            "logical_depth_after":
                int(
                    repair[
                        "logical_depth_after"
                    ]
                ),

            "max_weighted_degree_before":
                float(
                    repair[
                        "max_weighted_degree_before"
                    ]
                ),

            "max_weighted_degree_after":
                float(
                    repair[
                        "max_weighted_degree_after"
                    ]
                ),

            "swaps_before":
                swaps_before,

            "swaps_after":
                swaps_after,

            "delta_swaps":
                swaps_after
                - swaps_before,

            "routed_depth_before":
                depth_before,

            "routed_depth_after":
                depth_after,

            "delta_routed_depth":
                depth_after
                - depth_before,
        })


generalized_raw_routing = pd.DataFrame(
    generalized_raw_rows
)

print(
    "\nCompleted rows:",
    len(generalized_raw_routing)
)

print(
    "Failures:",
    len(generalized_routing_failures)
)


Routing generalized repair: system9.qasm | P1 -> P1_G2


2026-09-28 13:36:36,635 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 13:36:36,635 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 13:36:36,642 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 13:36:36,644 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 13:36:36,646 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 13:36:36,648 - INFO - Pass: BarrierBeforeFinalMeasurements - 2.01249 (ms)
2026-09-28 13:36:36,678 - INFO - Pass: SabreLayout - 30.02024 (ms)
2026-09-28 13:36:36,681 - INFO - Pass: CheckMap - 3.09300 (ms)
2026-09-28 13:36:36,681 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 13:36:36,688 - INFO - Pass: HighLevelSynthesis - 5.47647 (ms)
2026-09-28 13:36:36,688 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 13:36:36,692 - INFO - Pass: Depth - 1.15180 (ms)
2026-09-28 13:36:36,692 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:36,694 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:36,694 - INFO - Pas


Routing generalized repair: system9.qasm | P12 -> P12_G0


2026-09-28 13:36:37,484 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 13:36:37,484 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 13:36:37,484 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:37,484 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:37,484 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:37,493 - INFO - Total Transpile Time - 108.46591 (ms)
2026-09-28 13:36:37,509 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 13:36:37,509 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 13:36:37,517 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 13:36:37,517 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 13:36:37,517 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 13:36:37,517 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 13:36:37,559 - INFO - Pass: SabreLayout - 33.79774 (ms)
2026-09-28 13:36:37,559 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 13:36:37,559 - INFO - Pass: UnitaryS


Routing generalized repair: system9.qasm | P2 -> P2_G0


2026-09-28 13:36:38,260 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 13:36:38,263 - INFO - Pass: HighLevelSynthesis - 2.89536 (ms)
2026-09-28 13:36:38,265 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 13:36:38,268 - INFO - Pass: Depth - 2.75850 (ms)
2026-09-28 13:36:38,268 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:38,271 - INFO - Pass: Size - 2.63119 (ms)
2026-09-28 13:36:38,271 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:38,284 - INFO - Pass: Optimize1qGatesDecomposition - 13.33261 (ms)
2026-09-28 13:36:38,288 - INFO - Pass: CXCancellation - 1.34659 (ms)
2026-09-28 13:36:38,288 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 13:36:38,291 - INFO - Pass: Depth - 3.63421 (ms)
2026-09-28 13:36:38,291 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:38,293 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:38,295 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:38,301 - INFO - Pass: Optimize1qGatesDecomposition 

... (39 outputs left out)

2026-09-28 13:36:52,018 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,018 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:52,018 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,024 - INFO - Pass: Optimize1qGatesDecomposition - 6.23536 (ms)
2026-09-28 13:36:52,024 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 13:36:52,024 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 13:36:52,024 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 13:36:52,032 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,032 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:52,034 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,037 - INFO - Pass: Optimize1qGatesDecomposition - 2.36011 (ms)
2026-09-28 13:36:52,039 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 13:36:52,040 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 13:36:52,042 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 13:36:52,046 - INFO - Pass: FixedPoint - 0.00000 (ms)


Routing generalized repair: alu-bdd_288.qasm | P5 -> P5_G0


2026-09-28 13:36:52,467 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 13:36:52,470 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 13:36:52,470 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,472 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 13:36:52,472 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 13:36:52,474 - INFO - Total Transpile Time - 68.87102 (ms)
2026-09-28 13:36:52,484 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 13:36:52,484 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 13:36:52,484 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 13:36:52,492 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 13:36:52,492 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 13:36:52,492 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 13:36:52,509 - INFO - Pass: SabreLayout - 16.57796 (ms)
2026-09-28 13:36:52,511 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 13:36:52,511 - INFO - Pass: UnitarySy


Completed rows: 120
Failures: 0


In [70]:
# ------------------------------------------------------------
# Generalized intervention summaries
# ------------------------------------------------------------

generalized_intervention_rows = []

for (
    circuit_name,
    seed_plan_id,
    candidate_plan_id
), group in generalized_raw_routing.groupby(
    [
        "circuit",
        "seed_plan_id",
        "candidate_plan_id",
    ]
):

    repair = generalized_selected[
        (
            generalized_selected["circuit"]
            == circuit_name
        )
        &
        (
            generalized_selected["seed_plan_id"]
            == seed_plan_id
        )
        &
        (
            generalized_selected["candidate_plan_id"]
            == candidate_plan_id
        )
    ].iloc[0]

    mean_swaps_before = (
        group["swaps_before"].mean()
    )

    mean_swaps_after = (
        group["swaps_after"].mean()
    )

    mean_depth_before = (
        group[
            "routed_depth_before"
        ].mean()
    )

    mean_depth_after = (
        group[
            "routed_depth_after"
        ].mean()
    )

    swap_improvement_pct = (
        100
        *
        (
            mean_swaps_before
            - mean_swaps_after
        )
        /
        mean_swaps_before
    )

    depth_improvement_pct = (
        100
        *
        (
            mean_depth_before
            - mean_depth_after
        )
        /
        mean_depth_before
    )

    generalized_intervention_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            seed_plan_id,

        "candidate_plan_id":
            candidate_plan_id,

        "move_type":
            repair["move_type"],

        "logical_depth_before":
            int(
                repair[
                    "logical_depth_before"
                ]
            ),

        "logical_depth_after":
            int(
                repair[
                    "logical_depth_after"
                ]
            ),

        "relative_depth_change":
            float(
                repair[
                    "relative_depth_change"
                ]
            ),

        "depth_matched_5pct":
            abs(
                float(
                    repair[
                        "relative_depth_change"
                    ]
                )
            ) <= 0.05,

        "max_weighted_degree_before":
            float(
                repair[
                    "max_weighted_degree_before"
                ]
            ),

        "max_weighted_degree_after":
            float(
                repair[
                    "max_weighted_degree_after"
                ]
            ),

        "mean_swaps_before":
            mean_swaps_before,

        "mean_swaps_after":
            mean_swaps_after,

        "swap_improvement_pct":
            swap_improvement_pct,

        "mean_routed_depth_before":
            mean_depth_before,

        "mean_routed_depth_after":
            mean_depth_after,

        "depth_improvement_pct":
            depth_improvement_pct,

        "swap_better_seeds":
            int(
                (
                    group[
                        "delta_swaps"
                    ] < 0
                ).sum()
            ),

        "depth_better_seeds":
            int(
                (
                    group[
                        "delta_routed_depth"
                    ] < 0
                ).sum()
            ),
    })


generalized_interventions = pd.DataFrame(
    generalized_intervention_rows
)

display(
    generalized_interventions[
        [
            "circuit",
            "seed_plan_id",
            "move_type",
            "logical_depth_before",
            "logical_depth_after",
            "max_weighted_degree_before",
            "max_weighted_degree_after",
            "mean_swaps_before",
            "mean_swaps_after",
            "swap_improvement_pct",
            "mean_routed_depth_before",
            "mean_routed_depth_after",
            "depth_improvement_pct",
        ]
    ]
)

,circuit,seed_plan_id,move_type,logical_depth_before,logical_depth_after,max_weighted_degree_before,max_weighted_degree_after,mean_swaps_before,mean_swaps_after,swap_improvement_pct,mean_routed_depth_before,mean_routed_depth_after,depth_improvement_pct
0,alu-bdd_288.qasm,P0,one_sided_suffix_transfer,51,50,25.0,20.0,21.2,21.6,-1.886792,80.2,77.8,2.992519
1,alu-bdd_288.qasm,P5,one_sided_suffix_transfer,51,52,25.0,20.0,22.0,21.4,2.727273,82.6,79.8,3.389831
2,mini_alu_305.qasm,P1,two_sided_suffix_swap,103,103,44.0,40.0,43.0,43.0,0.000000,158.2,158.6,-0.252845
3,mini_alu_305.qasm,P11,two_sided_suffix_swap,102,102,43.0,39.0,44.2,45.2,-2.262443,157.4,158.6,-0.762389
4,mini_alu_305.qasm,P12,one_sided_suffix_transfer,103,103,44.0,33.0,41.8,43.0,-2.870813,154.8,157.2,-1.550388
5,mini_alu_305.qasm,P13,one_sided_suffix_transfer,98,101,54.0,41.0,45.4,47.0,-3.524229,159.6,165.2,-3.508772
6,mini_alu_305.qasm,P14,one_sided_suffix_transfer,98,101,42.0,41.0,45.2,45.6,-0.884956,159.4,162.2,-1.756587
7,mini_alu_305.qasm,P2,one_sided_suffix_transfer,94,94,39.0,31.0,41.6,43.6,-4.807692,156.0,152.6,2.179487
8,mini_alu_305.qasm,P3,two_sided_suffix_swap,103,103,44.0,40.0,44.4,45.4,-2.252252,157.4,157.2,0.127065
9,mini_alu_305.qasm,P4,one_sided_suffix_transfer,95,95,40.0,33.0,44.2,44.0,0.452489,156.0,152.6,2.179487


In [71]:
df = generalized_interventions

print(
    "Total generalized interventions:",
    len(df)
)

print(
    "\nLower mean SWAP:",
    int(
        (
            df[
                "mean_swaps_after"
            ]
            <
            df[
                "mean_swaps_before"
            ]
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "Lower mean routed depth:",
    int(
        (
            df[
                "mean_routed_depth_after"
            ]
            <
            df[
                "mean_routed_depth_before"
            ]
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "\nMean SWAP improvement:",
    df[
        "swap_improvement_pct"
    ].mean()
)

print(
    "Median SWAP improvement:",
    df[
        "swap_improvement_pct"
    ].median()
)

print(
    "\nMean depth improvement:",
    df[
        "depth_improvement_pct"
    ].mean()
)

print(
    "Median depth improvement:",
    df[
        "depth_improvement_pct"
    ].median()
)

Total generalized interventions: 24

Lower mean SWAP: 14 / 24
Lower mean routed depth: 15 / 24

Mean SWAP improvement: 1.2984390050746468
Median SWAP improvement: 0.6535947712418317

Mean depth improvement: 1.3903457271687187
Median depth improvement: 1.2837927488639003


In [74]:
# ------------------------------------------------------------
# Find Experiment-1 plan-summary dataframe safely
# ------------------------------------------------------------

required_cols = {
    "circuit",
    "plan_id",
    "width",
    "mean_swaps",
    "mean_routed_depth",
}

matches = []

# IMPORTANT:
# Take a snapshot first so globals() does not change during iteration
for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        if required_cols.issubset(
            set(obj.columns)
        ):

            matches.append(
                (name, obj.shape)
            )

print(
    "Matching dataframes:",
    matches
)

Matching dataframes: [('exp1_plans', (50, 17)), ('exp2_plans', (50, 41))]


In [78]:
exp1_plan = exp1_plans

print(
    "Experiment-1 plan summary shape:",
    exp1_plan.shape
)

display(
    exp1_plan[
        [
            "circuit",
            "plan_id",
            "width",
            "mean_swaps",
            "mean_routed_depth",
        ]
    ].head()
)

Experiment-1 plan summary shape: (50, 17)


,circuit,plan_id,width,mean_swaps,mean_routed_depth
0,system9.qasm,P0,7,88.2,294.6
1,system9.qasm,P1,7,94.2,306.2
2,system9.qasm,P10,7,87.2,296.6
3,system9.qasm,P11,7,91.2,288.4
4,system9.qasm,P12,7,88.8,298.4


In [79]:
# ------------------------------------------------------------
# ORACLE HEADROOM
#
# NOT a fair algorithmic baseline.
#
# Question:
# How far is the repaired plan from the best same-width
# GidNET plan already observed in Experiment 1?
# ------------------------------------------------------------

oracle_rows = []

for _, row in generalized_interventions.iterrows():

    circuit_name = (
        row["circuit"]
    )

    seed_plan_id = (
        row["seed_plan_id"]
    )

    # Width of this circuit/seed
    seed_width = int(
        exp2_plans[
            (
                exp2_plans["circuit"]
                == circuit_name
            )
            &
            (
                exp2_plans["plan_id"]
                == seed_plan_id
            )
        ].iloc[0]["width"]
    )

    # Same-width plans observed in Exp1
    same_width = (
        exp1_plan[
            (
                exp1_plan["circuit"]
                == circuit_name
            )
            &
            (
                exp1_plan["width"]
                == seed_width
            )
        ]
        .copy()
    )

    # Exclude current seed plan:
    # "what if GidNET had produced another plan?"
    alternatives = (
        same_width[
            same_width[
                "plan_id"
            ] != seed_plan_id
        ]
        .copy()
    )

    if len(alternatives) == 0:
        continue

    # Separate oracles for SWAP and depth
    best_swap_row = (
        alternatives
        .sort_values(
            "mean_swaps"
        )
        .iloc[0]
    )

    best_depth_row = (
        alternatives
        .sort_values(
            "mean_routed_depth"
        )
        .iloc[0]
    )

    repaired_swaps = float(
        row[
            "mean_swaps_after"
        ]
    )

    repaired_depth = float(
        row[
            "mean_routed_depth_after"
        ]
    )

    oracle_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            seed_plan_id,

        "repair_candidate_id":
            row[
                "candidate_plan_id"
            ],

        "original_mean_swaps":
            float(
                row[
                    "mean_swaps_before"
                ]
            ),

        "repaired_mean_swaps":
            repaired_swaps,

        "best_other_plan_by_swaps":
            best_swap_row[
                "plan_id"
            ],

        "best_other_mean_swaps":
            float(
                best_swap_row[
                    "mean_swaps"
                ]
            ),

        "repair_minus_oracle_swaps":
            repaired_swaps
            -
            float(
                best_swap_row[
                    "mean_swaps"
                ]
            ),

        "original_mean_depth":
            float(
                row[
                    "mean_routed_depth_before"
                ]
            ),

        "repaired_mean_depth":
            repaired_depth,

        "best_other_plan_by_depth":
            best_depth_row[
                "plan_id"
            ],

        "best_other_mean_depth":
            float(
                best_depth_row[
                    "mean_routed_depth"
                ]
            ),

        "repair_minus_oracle_depth":
            repaired_depth
            -
            float(
                best_depth_row[
                    "mean_routed_depth"
                ]
            ),
    })


repair_vs_oracle = pd.DataFrame(
    oracle_rows
)

display(
    repair_vs_oracle[
        [
            "circuit",
            "seed_plan_id",

            "original_mean_swaps",
            "repaired_mean_swaps",
            "best_other_plan_by_swaps",
            "best_other_mean_swaps",
            "repair_minus_oracle_swaps",

            "original_mean_depth",
            "repaired_mean_depth",
            "best_other_plan_by_depth",
            "best_other_mean_depth",
            "repair_minus_oracle_depth",
        ]
    ]
)

,circuit,seed_plan_id,original_mean_swaps,repaired_mean_swaps,best_other_plan_by_swaps,best_other_mean_swaps,repair_minus_oracle_swaps,original_mean_depth,repaired_mean_depth,best_other_plan_by_depth,best_other_mean_depth,repair_minus_oracle_depth
0,alu-bdd_288.qasm,P0,21.2,21.6,P2,21.0,0.6,80.2,77.8,P3,78.0,-0.2
1,alu-bdd_288.qasm,P5,22.0,21.4,P2,21.0,0.4,82.6,79.8,P3,78.0,1.8
2,mini_alu_305.qasm,P1,43.0,43.0,P2,41.6,1.4,158.2,158.6,P5,149.8,8.8
3,mini_alu_305.qasm,P11,44.2,45.2,P2,41.6,3.6,157.4,158.6,P5,149.8,8.8
4,mini_alu_305.qasm,P12,41.8,43.0,P2,41.6,1.4,154.8,157.2,P5,149.8,7.4
5,mini_alu_305.qasm,P13,45.4,47.0,P2,41.6,5.4,159.6,165.2,P5,149.8,15.4
6,mini_alu_305.qasm,P14,45.2,45.6,P2,41.6,4.0,159.4,162.2,P5,149.8,12.4
7,mini_alu_305.qasm,P2,41.6,43.6,P12,41.8,1.8,156.0,152.6,P5,149.8,2.8
8,mini_alu_305.qasm,P3,44.4,45.4,P2,41.6,3.8,157.4,157.2,P5,149.8,7.4
9,mini_alu_305.qasm,P4,44.2,44.0,P2,41.6,2.4,156.0,152.6,P5,149.8,2.8


In [80]:
print(
    "Repair reaches/beats best observed other plan in SWAP:",
    int(
        (
            repair_vs_oracle[
                "repair_minus_oracle_swaps"
            ] <= 0
        ).sum()
    ),
    "/",
    len(repair_vs_oracle)
)

print(
    "Repair reaches/beats best observed other plan in depth:",
    int(
        (
            repair_vs_oracle[
                "repair_minus_oracle_depth"
            ] <= 0
        ).sum()
    ),
    "/",
    len(repair_vs_oracle)
)

print(
    "\nMean remaining SWAP gap to oracle:",
    repair_vs_oracle[
        "repair_minus_oracle_swaps"
    ].mean()
)

print(
    "Mean remaining depth gap to oracle:",
    repair_vs_oracle[
        "repair_minus_oracle_depth"
    ].mean()
)

Repair reaches/beats best observed other plan in SWAP: 1 / 24
Repair reaches/beats best observed other plan in depth: 4 / 24

Mean remaining SWAP gap to oracle: 3.949999999999998
Mean remaining depth gap to oracle: 6.524999999999992


In [81]:
# ------------------------------------------------------------
# Helpers for independent GidNET restarts
# ------------------------------------------------------------

import time


def get_original_active_qubit_indices(circuit):
    """
    Same active-qubit interpretation used earlier:
    ignore barriers and measurements when determining
    the logical qubits participating in the computation.
    """

    active = set()

    for item in circuit.data:

        if item.operation.name in {
            "barrier",
            "measure",
        }:
            continue

        for q in item.qubits:
            active.add(
                circuit.find_bit(q).index
            )

    return sorted(active)


def extract_active_reuse_chains(
    circuit,
    raw_reuse_sequences,
):
    """
    GidNET may contain declared-but-inactive qubits
    (and, in this implementation, additional matrix indices).

    Keep only original ACTIVE logical-qubit indices and
    remove empty chains.
    """

    active_set = set(
        get_original_active_qubit_indices(
            circuit
        )
    )

    active_chains = []

    for chain in raw_reuse_sequences:

        filtered = tuple(
            q
            for q in chain
            if q in active_set
        )

        if filtered:
            active_chains.append(
                filtered
            )

    # Sanity: every active logical qubit exactly once
    flat = [
        q
        for chain in active_chains
        for q in chain
    ]

    if (
        len(flat) != len(set(flat))
        or
        set(flat) != active_set
    ):
        raise ValueError(
            "Restart plan does not represent every "
            "active logical qubit exactly once."
        )

    return active_chains

In [82]:
# ------------------------------------------------------------
# Generate ONE independent same-width GidNET restart
# ------------------------------------------------------------

def generate_same_width_restart(
    circuit,
    target_width,
    first_seed,
    max_attempts=50,
):
    """
    Generate one GidNET restart whose ACTIVE reuse width
    equals target_width.

    Width-miss attempts are recorded but not physically routed.
    The first same-width result is accepted even if it reproduces
    the original plan.
    """

    attempts = []

    total_start = time.perf_counter()

    for offset in range(max_attempts):

        gidnet_seed = (
            first_seed + offset
        )

        random.seed(
            gidnet_seed
        )

        np.random.seed(
            gidnet_seed
        )

        t0 = time.perf_counter()

        g = GidNET(
            copy.deepcopy(circuit)
        )

        dynamic = (
            g.compile_to_dynamic_circuit(
                iterations=20,
                draw=False,
            )
        )

        runtime = (
            time.perf_counter()
            - t0
        )

        active_chains = (
            extract_active_reuse_chains(
                circuit,
                g.qubit_reuse_sequences,
            )
        )

        active_width = len(
            active_chains
        )

        attempts.append({
            "gidnet_seed":
                gidnet_seed,

            "active_width":
                active_width,

            "generation_runtime_sec":
                runtime,
        })

        if active_width != target_width:
            continue

        dynamic_norm = (
            normalize_active_reset(
                dynamic
            )
        )

        return {
            "success":
                True,

            "gidnet_seed":
                gidnet_seed,

            "active_chains":
                active_chains,

            "plan_signature":
                canonical_plan_signature(
                    active_chains
                ),

            "active_width":
                active_width,

            "normalized_circuit":
                dynamic_norm,

            "normalized_logical_depth":
                dynamic_norm.depth(),

            "num_generation_attempts":
                len(attempts),

            "total_generation_runtime_sec":
                time.perf_counter()
                - total_start,

            "attempts":
                attempts,
        }

    return {
        "success":
            False,

        "num_generation_attempts":
            len(attempts),

        "total_generation_runtime_sec":
            time.perf_counter()
            - total_start,

        "attempts":
            attempts,
    }

In [83]:
# ------------------------------------------------------------
# Sanity check: one held-out GidNET restart
# ------------------------------------------------------------

test_start = generalized_selected[
    (
        generalized_selected[
            "circuit"
        ] == "system9.qasm"
    )
    &
    (
        generalized_selected[
            "seed_plan_id"
        ] == "P1"
    )
].iloc[0]

test_qc = circuit_cache[
    "system9.qasm"
]

test_restart = (
    generate_same_width_restart(
        circuit=test_qc,
        target_width=int(
            test_start["width_before"]
        ),
        first_seed=1000,
        max_attempts=50,
    )
)

print(
    "Success:",
    test_restart["success"]
)

print(
    "Generation attempts:",
    test_restart[
        "num_generation_attempts"
    ]
)

if test_restart["success"]:

    print(
        "GidNET seed:",
        test_restart["gidnet_seed"]
    )

    print(
        "Active width:",
        test_restart["active_width"]
    )

    print(
        "Normalized logical depth:",
        test_restart[
            "normalized_logical_depth"
        ]
    )

    print(
        "\nRestart chains:"
    )

    for chain in test_restart[
        "active_chains"
    ]:
        print(" ", chain)

    print(
        "\nRestart signature:"
    )

    print(
        test_restart[
            "plan_signature"
        ]
    )

Success: True
Generation attempts: 1
GidNET seed: 1000
Active width: 7
Normalized logical depth: 179

Restart chains:
  (0, 2, 4, 6, 8)
  (5, 7)
  (1,)
  (3,)
  (9,)
  (10,)
  (11,)

Restart signature:
((0, 2, 4, 6, 8), (1,), (3,), (5, 7), (9,), (10,), (11,))


In [84]:
# ------------------------------------------------------------
# Experiment 3B — Budget 1
# Generate ONE independent same-width GidNET restart
# for each of the 24 repair-eligible starting plans
# ------------------------------------------------------------

restart_generation_rows = []
restart_objects = {}

starts_for_budget = (
    generalized_selected
    .sort_values(
        [
            "circuit",
            "seed_plan_id",
        ]
    )
    .reset_index(drop=True)
)

for i, start in starts_for_budget.iterrows():

    circuit_name = start["circuit"]
    seed_plan_id = start["seed_plan_id"]

    qc = circuit_cache[circuit_name]

    target_width = int(
        start["width_before"]
    )

    # Separate deterministic seed block for every starting plan.
    # Each block has room for max_attempts=50.
    first_restart_seed = (
        10000 + i * 100
    )

    print(
        f"\nGenerating restart for "
        f"{circuit_name} | {seed_plan_id}"
    )

    result = generate_same_width_restart(
        circuit=qc,
        target_width=target_width,
        first_seed=first_restart_seed,
        max_attempts=50,
    )

    if not result["success"]:

        restart_generation_rows.append({
            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "success":
                False,

            "first_restart_seed":
                first_restart_seed,

            "num_generation_attempts":
                result[
                    "num_generation_attempts"
                ],

            "generation_runtime_sec":
                result[
                    "total_generation_runtime_sec"
                ],
        })

        print("  FAILED to recover target width")
        continue

    # --------------------------------------------------------
    # Recover starting plan signature
    # --------------------------------------------------------

    seed_row = exp2_plans[
        (
            exp2_plans["circuit"]
            == circuit_name
        )
        &
        (
            exp2_plans["plan_id"]
            == seed_plan_id
        )
    ].iloc[0]

    seed_chains = [
        tuple(chain)
        for chain in parse_saved_structure(
            seed_row["reuse_chains"]
        )
    ]

    seed_signature = (
        canonical_plan_signature(
            seed_chains
        )
    )

    same_as_start = (
        result["plan_signature"]
        == seed_signature
    )

    key = (
        circuit_name,
        seed_plan_id
    )

    restart_objects[key] = result

    restart_generation_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            seed_plan_id,

        "success":
            True,

        "first_restart_seed":
            first_restart_seed,

        "restart_gidnet_seed":
            result["gidnet_seed"],

        "target_width":
            target_width,

        "restart_width":
            result["active_width"],

        "restart_logical_depth":
            result[
                "normalized_logical_depth"
            ],

        "same_plan_as_start":
            same_as_start,

        "num_generation_attempts":
            result[
                "num_generation_attempts"
            ],

        "generation_runtime_sec":
            result[
                "total_generation_runtime_sec"
            ],

        "restart_signature":
            result[
                "plan_signature"
            ],
    })

    print(
        f"  seed={result['gidnet_seed']} | "
        f"width={result['active_width']} | "
        f"depth={result['normalized_logical_depth']} | "
        f"same_as_start={same_as_start}"
    )


restart_generation = pd.DataFrame(
    restart_generation_rows
)

print(
    "\nSuccessful restart candidates:",
    int(
        restart_generation[
            "success"
        ].sum()
    ),
    "/",
    len(restart_generation)
)

print(
    "Restart reproduced original plan:",
    int(
        restart_generation.get(
            "same_plan_as_start",
            pd.Series(dtype=bool)
        )
        .fillna(False)
        .sum()
    )
)

print(
    "Mean GidNET attempts needed:",
    restart_generation[
        "num_generation_attempts"
    ].mean()
)


Generating restart for alu-bdd_288.qasm | P0
  seed=10000 | width=5 | depth=50 | same_as_start=False

Generating restart for alu-bdd_288.qasm | P5
  seed=10100 | width=5 | depth=50 | same_as_start=False

Generating restart for mini_alu_305.qasm | P1
  seed=10200 | width=6 | depth=94 | same_as_start=False

Generating restart for mini_alu_305.qasm | P11
  seed=10300 | width=6 | depth=95 | same_as_start=False

Generating restart for mini_alu_305.qasm | P12
  seed=10400 | width=6 | depth=105 | same_as_start=False

Generating restart for mini_alu_305.qasm | P13
  seed=10500 | width=6 | depth=101 | same_as_start=False

Generating restart for mini_alu_305.qasm | P14
  seed=10600 | width=6 | depth=103 | same_as_start=False

Generating restart for mini_alu_305.qasm | P2
  seed=10700 | width=6 | depth=95 | same_as_start=False

Generating restart for mini_alu_305.qasm | P3
  seed=10800 | width=6 | depth=103 | same_as_start=False

Generating restart for mini_alu_305.qasm | P4
  seed=10900 | width

In [85]:
# ------------------------------------------------------------
# Budget 1 — Physical evaluation of restart candidates
# ------------------------------------------------------------

restart_raw_rows = []
restart_routing_failures = []

for _, info in restart_generation.iterrows():

    if not info["success"]:
        continue

    circuit_name = info["circuit"]
    seed_plan_id = info["seed_plan_id"]

    key = (
        circuit_name,
        seed_plan_id
    )

    restart = restart_objects[key]

    restart_norm = (
        restart[
            "normalized_circuit"
        ]
    )

    print(
        f"\nRouting restart: "
        f"{circuit_name} | {seed_plan_id} | "
        f"GidNET seed={restart['gidnet_seed']}"
    )

    # --------------------------------------------------------
    # Original starting-plan results from Experiment 1
    # --------------------------------------------------------

    original_raw = (
        exp1_raw[
            (
                exp1_raw["circuit"]
                == circuit_name
            )
            &
            (
                exp1_raw["plan_id"]
                == seed_plan_id
            )
        ]
        .sort_values("routing_seed")
        .copy()
    )

    assert (
        original_raw[
            "routing_seed"
        ].tolist()
        ==
        ROUTING_SEEDS
    )

    original_by_seed = (
        original_raw
        .set_index("routing_seed")
    )

    # --------------------------------------------------------
    # Same paired SABRE seeds
    # --------------------------------------------------------

    for routing_seed in ROUTING_SEEDS:

        try:

            routed = transpile(
                restart_norm,
                coupling_map=TARGET_COUPLING,
                routing_method="sabre",
                layout_method="sabre",
                optimization_level=1,
                seed_transpiler=routing_seed,
            )

        except Exception as e:

            restart_routing_failures.append({
                "circuit":
                    circuit_name,

                "seed_plan_id":
                    seed_plan_id,

                "restart_gidnet_seed":
                    restart["gidnet_seed"],

                "routing_seed":
                    routing_seed,

                "error":
                    repr(e),
            })

            continue

        before = original_by_seed.loc[
            routing_seed
        ]

        swaps_before = int(
            before["swap_count"]
        )

        depth_before = int(
            before["routed_depth"]
        )

        swaps_after = int(
            routed.count_ops().get(
                "swap",
                0
            )
        )

        depth_after = int(
            routed.depth()
        )

        restart_raw_rows.append({

            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "restart_gidnet_seed":
                restart["gidnet_seed"],

            "same_plan_as_start":
                bool(
                    info[
                        "same_plan_as_start"
                    ]
                ),

            "routing_seed":
                routing_seed,

            "restart_logical_depth":
                restart[
                    "normalized_logical_depth"
                ],

            "swaps_before":
                swaps_before,

            "swaps_after":
                swaps_after,

            "delta_swaps":
                swaps_after
                - swaps_before,

            "routed_depth_before":
                depth_before,

            "routed_depth_after":
                depth_after,

            "delta_routed_depth":
                depth_after
                - depth_before,
        })


restart_raw = pd.DataFrame(
    restart_raw_rows
)

print(
    "\nCompleted restart routing rows:",
    len(restart_raw)
)

print(
    "Routing failures:",
    len(
        restart_routing_failures
    )
)

2026-09-28 16:20:54,458 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 16:20:54,460 - INFO - Pass: UnitarySynthesis - 2.00891 (ms)
2026-09-28 16:20:54,460 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 16:20:54,462 - INFO - Pass: Unroll3qOrMore - 0.00000 (ms)
2026-09-28 16:20:54,463 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 16:20:54,463 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 16:20:54,479 - INFO - Pass: SabreLayout - 16.22796 (ms)
2026-09-28 16:20:54,487 - INFO - Pass: CheckMap - 0.47326 (ms)
2026-09-28 16:20:54,487 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 16:20:54,490 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 16:20:54,496 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 16:20:54,498 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:20:54,500 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:54,503 - INFO - Pass: Size - 0.85664 (ms)
2026-09-28 16:20:54,503 - INFO - Pas


Routing restart: alu-bdd_288.qasm | P0 | GidNET seed=10000


2026-09-28 16:20:54,680 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:20:54,680 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:20:54,680 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:20:54,680 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:54,689 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:20:54,689 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:54,691 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 16:20:54,691 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:20:54,696 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:20:54,696 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:20:54,703 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:54,705 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:20:54,706 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:54,706 - INFO - Total Transpile Time - 260.17833 (ms)
2026-09-28 16:20:54,723 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
20


Routing restart: alu-bdd_288.qasm | P5 | GidNET seed=10100


2026-09-28 16:20:55,233 - INFO - Pass: HighLevelSynthesis - 2.00868 (ms)
2026-09-28 16:20:55,235 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 16:20:55,235 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:20:55,237 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:55,239 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:20:55,239 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:55,239 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 16:20:55,239 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:20:55,247 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 16:20:55,255 - INFO - Pass: CXCancellation - 8


Routing restart: mini_alu_305.qasm | P1 | GidNET seed=10200


... (40 outputs left out)

2026-09-28 16:21:05,739 - INFO - Pass: Optimize1qGatesDecomposition - 14.83035 (ms)
2026-09-28 16:21:05,747 - INFO - Pass: CXCancellation - 7.42841 (ms)
2026-09-28 16:21:05,747 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:05,747 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:05,747 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:05,747 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:05,755 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:05,757 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:05,774 - INFO - Total Transpile Time - 105.544


Routing restart: system9.qasm | P8 | GidNET seed=12300


2026-09-28 16:21:06,306 - INFO - Pass: Optimize1qGatesDecomposition - 13.69596 (ms)
2026-09-28 16:21:06,306 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:06,312 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:06,312 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:06,312 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:06,319 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:06,320 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: Optimize1qGatesDecomposition - 5.64504 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:06,328 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:06,339 - INFO - Total Transpile Time - 106.434


Completed restart routing rows: 120
Routing failures: 0


In [86]:
# ------------------------------------------------------------
# Budget 1 — Physical evaluation of restart candidates
# ------------------------------------------------------------

restart_raw_rows = []
restart_routing_failures = []

for _, info in restart_generation.iterrows():

    if not info["success"]:
        continue

    circuit_name = info["circuit"]
    seed_plan_id = info["seed_plan_id"]

    key = (
        circuit_name,
        seed_plan_id
    )

    restart = restart_objects[key]

    restart_norm = (
        restart[
            "normalized_circuit"
        ]
    )

    print(
        f"\nRouting restart: "
        f"{circuit_name} | {seed_plan_id} | "
        f"GidNET seed={restart['gidnet_seed']}"
    )

    # --------------------------------------------------------
    # Original starting-plan results from Experiment 1
    # --------------------------------------------------------

    original_raw = (
        exp1_raw[
            (
                exp1_raw["circuit"]
                == circuit_name
            )
            &
            (
                exp1_raw["plan_id"]
                == seed_plan_id
            )
        ]
        .sort_values("routing_seed")
        .copy()
    )

    assert (
        original_raw[
            "routing_seed"
        ].tolist()
        ==
        ROUTING_SEEDS
    )

    original_by_seed = (
        original_raw
        .set_index("routing_seed")
    )

    # --------------------------------------------------------
    # Same paired SABRE seeds
    # --------------------------------------------------------

    for routing_seed in ROUTING_SEEDS:

        try:

            routed = transpile(
                restart_norm,
                coupling_map=TARGET_COUPLING,
                routing_method="sabre",
                layout_method="sabre",
                optimization_level=1,
                seed_transpiler=routing_seed,
            )

        except Exception as e:

            restart_routing_failures.append({
                "circuit":
                    circuit_name,

                "seed_plan_id":
                    seed_plan_id,

                "restart_gidnet_seed":
                    restart["gidnet_seed"],

                "routing_seed":
                    routing_seed,

                "error":
                    repr(e),
            })

            continue

        before = original_by_seed.loc[
            routing_seed
        ]

        swaps_before = int(
            before["swap_count"]
        )

        depth_before = int(
            before["routed_depth"]
        )

        swaps_after = int(
            routed.count_ops().get(
                "swap",
                0
            )
        )

        depth_after = int(
            routed.depth()
        )

        restart_raw_rows.append({

            "circuit":
                circuit_name,

            "seed_plan_id":
                seed_plan_id,

            "restart_gidnet_seed":
                restart["gidnet_seed"],

            "same_plan_as_start":
                bool(
                    info[
                        "same_plan_as_start"
                    ]
                ),

            "routing_seed":
                routing_seed,

            "restart_logical_depth":
                restart[
                    "normalized_logical_depth"
                ],

            "swaps_before":
                swaps_before,

            "swaps_after":
                swaps_after,

            "delta_swaps":
                swaps_after
                - swaps_before,

            "routed_depth_before":
                depth_before,

            "routed_depth_after":
                depth_after,

            "delta_routed_depth":
                depth_after
                - depth_before,
        })


restart_raw = pd.DataFrame(
    restart_raw_rows
)

print(
    "\nCompleted restart routing rows:",
    len(restart_raw)
)

print(
    "Routing failures:",
    len(
        restart_routing_failures
    )
)

2026-09-28 16:21:47,263 - INFO - Pass: ContainsInstruction - 0.00000 (ms)
2026-09-28 16:21:47,265 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 16:21:47,267 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 16:21:47,270 - INFO - Pass: Unroll3qOrMore - 2.40898 (ms)
2026-09-28 16:21:47,270 - INFO - Pass: SetLayout - 0.00000 (ms)
2026-09-28 16:21:47,272 - INFO - Pass: BarrierBeforeFinalMeasurements - 0.00000 (ms)
2026-09-28 16:21:47,292 - INFO - Pass: SabreLayout - 14.72092 (ms)
2026-09-28 16:21:47,295 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 16:21:47,297 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 16:21:47,301 - INFO - Pass: HighLevelSynthesis - 2.02036 (ms)
2026-09-28 16:21:47,303 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 16:21:47,305 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:47,305 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,305 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:47,305 - INFO - Pas


Routing restart: alu-bdd_288.qasm | P0 | GidNET seed=10000


2026-09-28 16:21:47,421 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:47,422 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,433 - INFO - Pass: Optimize1qGatesDecomposition - 3.24750 (ms)
2026-09-28 16:21:47,439 - INFO - Pass: CXCancellation - 0.81086 (ms)
2026-09-28 16:21:47,439 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:47,442 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:47,445 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,447 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:47,447 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,447 - INFO - Pass: Optimize1qGatesDecomposition - 0.00000 (ms)
2026-09-28 16:21:47,455 - INFO - Pass: CXCancellation - 0.85044 (ms)
2026-09-28 16:21:47,457 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:47,457 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:47,462 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,462 - INFO - Pass: Size - 0.00000 (ms)
2026-


Routing restart: alu-bdd_288.qasm | P5 | GidNET seed=10100


2026-09-28 16:21:47,920 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:47,920 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,922 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:47,922 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,929 - INFO - Pass: Optimize1qGatesDecomposition - 6.49500 (ms)
2026-09-28 16:21:47,930 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:47,932 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:47,933 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:47,933 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,933 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:47,933 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:47,939 - INFO - Pass: Optimize1qGatesDecomposition - 0.68665 (ms)
2026-09-28 16:21:47,943 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:47,945 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:47,947 - INFO - Pass: Depth - 0.00000 (ms)
2026


Routing restart: mini_alu_305.qasm | P1 | GidNET seed=10200


... (40 outputs left out)

2026-09-28 16:21:59,720 - INFO - Pass: CheckMap - 13.92102 (ms)
2026-09-28 16:21:59,721 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 16:21:59,723 - INFO - Pass: HighLevelSynthesis - 0.00000 (ms)
2026-09-28 16:21:59,727 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 16:21:59,727 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:59,727 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:59,727 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:59,727 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:59,744 - INFO - Pass: Optimize1qGatesDecomposition - 17.09938 (ms)
2026-09-28 16:21:59,744 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:21:59,752 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:21:59,755 - INFO - Pass: Depth - 0.00000 (ms)
2026-09-28 16:21:59,757 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:21:59,757 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:21:59,757 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026


Routing restart: system9.qasm | P8 | GidNET seed=12300


2026-09-28 16:22:00,356 - INFO - Pass: SabreLayout - 26.16167 (ms)
2026-09-28 16:22:00,356 - INFO - Pass: CheckMap - 0.00000 (ms)
2026-09-28 16:22:00,356 - INFO - Pass: UnitarySynthesis - 0.00000 (ms)
2026-09-28 16:22:00,366 - INFO - Pass: HighLevelSynthesis - 9.55892 (ms)
2026-09-28 16:22:00,366 - INFO - Pass: BasisTranslator - 0.00000 (ms)
2026-09-28 16:22:00,370 - INFO - Pass: Depth - 1.74856 (ms)
2026-09-28 16:22:00,370 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:22:00,371 - INFO - Pass: Size - 0.00000 (ms)
2026-09-28 16:22:00,373 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:22:00,389 - INFO - Pass: Optimize1qGatesDecomposition - 15.84840 (ms)
2026-09-28 16:22:00,390 - INFO - Pass: CXCancellation - 0.00000 (ms)
2026-09-28 16:22:00,390 - INFO - Pass: GatesInBasis - 0.00000 (ms)
2026-09-28 16:22:00,395 - INFO - Pass: Depth - 5.05924 (ms)
2026-09-28 16:22:00,395 - INFO - Pass: FixedPoint - 0.00000 (ms)
2026-09-28 16:22:00,395 - INFO - Pass: Size - 0.00000 (ms)
202


Completed restart routing rows: 120
Routing failures: 0


In [87]:
# ------------------------------------------------------------
# Restart candidate summary
# ------------------------------------------------------------

restart_summary_rows = []

for (
    circuit_name,
    seed_plan_id
), group in restart_raw.groupby(
    [
        "circuit",
        "seed_plan_id",
    ]
):

    mean_before_swaps = (
        group["swaps_before"].mean()
    )

    mean_after_swaps = (
        group["swaps_after"].mean()
    )

    mean_before_depth = (
        group[
            "routed_depth_before"
        ].mean()
    )

    mean_after_depth = (
        group[
            "routed_depth_after"
        ].mean()
    )

    restart_summary_rows.append({

        "circuit":
            circuit_name,

        "seed_plan_id":
            seed_plan_id,

        "restart_gidnet_seed":
            int(
                group[
                    "restart_gidnet_seed"
                ].iloc[0]
            ),

        "same_plan_as_start":
            bool(
                group[
                    "same_plan_as_start"
                ].iloc[0]
            ),

        "restart_logical_depth":
            int(
                group[
                    "restart_logical_depth"
                ].iloc[0]
            ),

        "original_mean_swaps":
            mean_before_swaps,

        "restart_mean_swaps":
            mean_after_swaps,

        "restart_swap_improvement_pct":
            100
            *
            (
                mean_before_swaps
                - mean_after_swaps
            )
            /
            mean_before_swaps,

        "original_mean_depth":
            mean_before_depth,

        "restart_mean_depth":
            mean_after_depth,

        "restart_depth_improvement_pct":
            100
            *
            (
                mean_before_depth
                - mean_after_depth
            )
            /
            mean_before_depth,
    })


restart_summary = pd.DataFrame(
    restart_summary_rows
)

display(restart_summary)

,circuit,seed_plan_id,restart_gidnet_seed,same_plan_as_start,restart_logical_depth,original_mean_swaps,restart_mean_swaps,restart_swap_improvement_pct,original_mean_depth,restart_mean_depth,restart_depth_improvement_pct
0,alu-bdd_288.qasm,P0,10000,False,50,21.2,23.2,-9.433962,80.2,79.6,0.748130
1,alu-bdd_288.qasm,P5,10100,False,50,22.0,23.2,-5.454545,82.6,79.6,3.631961
2,mini_alu_305.qasm,P1,10200,False,94,43.0,42.6,0.930233,158.2,153.6,2.907712
3,mini_alu_305.qasm,P11,10300,False,95,44.2,45.0,-1.809955,157.4,155.2,1.397713
4,mini_alu_305.qasm,P12,10400,False,105,41.8,42.2,-0.956938,154.8,161.2,-4.134367
5,mini_alu_305.qasm,P13,10500,False,101,45.4,46.4,-2.202643,159.6,164.0,-2.756892
6,mini_alu_305.qasm,P14,10600,False,103,45.2,44.0,2.654867,159.4,159.2,0.125471
7,mini_alu_305.qasm,P2,10700,False,95,41.6,44.8,-7.692308,156.0,154.6,0.897436
8,mini_alu_305.qasm,P3,10800,False,103,44.4,42.0,5.405405,157.4,158.4,-0.635324
9,mini_alu_305.qasm,P4,10900,False,103,44.2,44.4,-0.452489,156.0,159.8,-2.435897


In [88]:
# ------------------------------------------------------------
# Budget=1 direct comparison:
# one generalized repair vs one independent GidNET restart
# ------------------------------------------------------------

budget1_comparison = (
    generalized_interventions[
        [
            "circuit",
            "seed_plan_id",

            "mean_swaps_before",
            "mean_swaps_after",
            "swap_improvement_pct",

            "mean_routed_depth_before",
            "mean_routed_depth_after",
            "depth_improvement_pct",
        ]
    ]
    .rename(
        columns={
            "mean_swaps_after":
                "repair_mean_swaps",

            "swap_improvement_pct":
                "repair_swap_improvement_pct",

            "mean_routed_depth_after":
                "repair_mean_depth",

            "depth_improvement_pct":
                "repair_depth_improvement_pct",
        }
    )
    .merge(
        restart_summary,
        on=[
            "circuit",
            "seed_plan_id",
        ],
        how="inner",
    )
)


# Positive value means REPAIR is better than restart.
budget1_comparison[
    "repair_swap_advantage_over_restart"
] = (
    budget1_comparison[
        "restart_mean_swaps"
    ]
    -
    budget1_comparison[
        "repair_mean_swaps"
    ]
)

budget1_comparison[
    "repair_depth_advantage_over_restart"
] = (
    budget1_comparison[
        "restart_mean_depth"
    ]
    -
    budget1_comparison[
        "repair_mean_depth"
    ]
)


display(
    budget1_comparison[
        [
            "circuit",
            "seed_plan_id",

            "mean_swaps_before",
            "repair_mean_swaps",
            "restart_mean_swaps",
            "repair_swap_advantage_over_restart",

            "mean_routed_depth_before",
            "repair_mean_depth",
            "restart_mean_depth",
            "repair_depth_advantage_over_restart",

            "same_plan_as_start",
        ]
    ]
)

,circuit,seed_plan_id,mean_swaps_before,repair_mean_swaps,restart_mean_swaps,repair_swap_advantage_over_restart,mean_routed_depth_before,repair_mean_depth,restart_mean_depth,repair_depth_advantage_over_restart,same_plan_as_start
0,alu-bdd_288.qasm,P0,21.2,21.6,23.2,1.6,80.2,77.8,79.6,1.8,False
1,alu-bdd_288.qasm,P5,22.0,21.4,23.2,1.8,82.6,79.8,79.6,-0.2,False
2,mini_alu_305.qasm,P1,43.0,43.0,42.6,-0.4,158.2,158.6,153.6,-5.0,False
3,mini_alu_305.qasm,P11,44.2,45.2,45.0,-0.2,157.4,158.6,155.2,-3.4,False
4,mini_alu_305.qasm,P12,41.8,43.0,42.2,-0.8,154.8,157.2,161.2,4.0,False
5,mini_alu_305.qasm,P13,45.4,47.0,46.4,-0.6,159.6,165.2,164.0,-1.2,False
6,mini_alu_305.qasm,P14,45.2,45.6,44.0,-1.6,159.4,162.2,159.2,-3.0,False
7,mini_alu_305.qasm,P2,41.6,43.6,44.8,1.2,156.0,152.6,154.6,2.0,False
8,mini_alu_305.qasm,P3,44.4,45.4,42.0,-3.4,157.4,157.2,158.4,1.2,False
9,mini_alu_305.qasm,P4,44.2,44.0,44.4,0.4,156.0,152.6,159.8,7.2,False


In [89]:
# ------------------------------------------------------------
# Budget=1 verdict
# ------------------------------------------------------------

df = budget1_comparison

print(
    "Budget=1 comparisons:",
    len(df)
)

print(
    "\nREPAIR beats RESTART in mean SWAP:",
    int(
        (
            df[
                "repair_swap_advantage_over_restart"
            ] > 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "RESTART beats REPAIR in mean SWAP:",
    int(
        (
            df[
                "repair_swap_advantage_over_restart"
            ] < 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "Tie in mean SWAP:",
    int(
        np.isclose(
            df[
                "repair_swap_advantage_over_restart"
            ],
            0
        ).sum()
    ),
    "/",
    len(df)
)


print(
    "\nREPAIR beats RESTART in mean routed depth:",
    int(
        (
            df[
                "repair_depth_advantage_over_restart"
            ] > 0
        ).sum()
    ),
    "/",
    len(df)
)

print(
    "RESTART beats REPAIR in mean routed depth:",
    int(
        (
            df[
                "repair_depth_advantage_over_restart"
            ] < 0
        ).sum()
    ),
    "/",
    len(df)
)


print(
    "\nMean repair SWAP improvement (%):",
    df[
        "repair_swap_improvement_pct"
    ].mean()
)

print(
    "Mean restart SWAP improvement (%):",
    df[
        "restart_swap_improvement_pct"
    ].mean()
)


print(
    "\nMean repair depth improvement (%):",
    df[
        "repair_depth_improvement_pct"
    ].mean()
)

print(
    "Mean restart depth improvement (%):",
    df[
        "restart_depth_improvement_pct"
    ].mean()
)

Budget=1 comparisons: 24

REPAIR beats RESTART in mean SWAP: 9 / 24
RESTART beats REPAIR in mean SWAP: 14 / 24
Tie in mean SWAP: 1 / 24

REPAIR beats RESTART in mean routed depth: 15 / 24
RESTART beats REPAIR in mean routed depth: 9 / 24

Mean repair SWAP improvement (%): 1.2984390050746468
Mean restart SWAP improvement (%): 2.6844709304386813

Mean repair depth improvement (%): 1.3903457271687187
Mean restart depth improvement (%): 0.9578962299084964
